# 701 — Connectivity Analysis

## Objective

Execute the prospectively frozen Phase 7 perturbational-connectivity analysis using the stable Notebook 700 handoffs.

This notebook performs the first observed program–perturbagen connectivity calculations in Phase 7. It preserves the frozen BING query representation, resistance-like orientation, signature → condition → timepoint → cell-line → lineage → perturbagen hierarchy, conditional assignment-specific null, global Benjamini–Yekutieli family, and prespecified sensitivity/diagnostic boundaries.

## Execution boundary

Notebook 701 must not use Phase 5 or Phase 6 evidence to define, filter, rescue, or reorder primary results.

The primary long-running Monte Carlo cell is explicitly marked **LONG RUN**. It is checkpointed and restartable. Re-running that cell resumes from the highest fully committed checkpoint rather than restarting completed replicates.

The outputs are computational perturbational hypotheses. They do not establish therapeutic reversal, efficacy, causal mechanism, validated targets, or clinical predictiveness.


In [ ]:
# =============================================================================
# Imports and frozen Notebook 701 constants
# =============================================================================

import hashlib
import h5py
import json
import os
import platform
import shutil
import sys
import tempfile
import time
from importlib.metadata import version as package_version
from pathlib import Path

import numpy as np
import pandas as pd
from threadpoolctl import threadpool_info

from pancancer_epigenetics.utils.artifact_registry import (
    load_artifact_registry,
    resolve_artifact_path,
    validate_artifact_registry,
)
from pancancer_epigenetics.utils.file_checks import (
    calculate_sha256,
)
from pancancer_epigenetics.utils.paths import (
    Paths,
    project_relative_path,
)

PROGRAM_IDS = (
    "CONSENSUS_TX_01",
    "CONSENSUS_TX_02",
    "CONSENSUS_TX_03",
)

RNG_ENTROPY = 701
RNG_NAMESPACES = {
    "primary_conditional_null": 1,
    "competitive_null_sensitivity": 2,
    "outcome_blind_benchmark": 3,
}

PRIMARY_FAMILY_SIZE = 23742
PRIMARY_BY_Q = 0.05
MC_EXCEEDANCE_TARGET = 20
FINAL_BMAX = 5058108
MC_BATCH_SIZE = 20
CHECKPOINT_INTERVAL = 1000

NUMERICAL_RTOL = 1e-10
NUMERICAL_ATOL = 1e-12
SIGNATURE_BLOCK_SIZE = 4096

PRIMARY_MIN_CELL_LINES = 5
PRIMARY_MIN_LINEAGES = 4

print("Notebook 701 constants: FROZEN")
print("Programs:", PROGRAM_IDS)
print("Primary family:", f"{PRIMARY_FAMILY_SIZE:,}")
print("h:", MC_EXCEEDANCE_TARGET)
print("Bmax:", f"{FINAL_BMAX:,}")
print("MC batch size:", MC_BATCH_SIZE)


In [ ]:
# =============================================================================
# Resolve frozen Notebook 700 and LINCS inputs
# =============================================================================

artifact_registry = load_artifact_registry()

INPUT_ARTIFACT_IDS = (
    "phase7.700.program_query_definitions",
    "phase7.700.null_stratification_manifest",
    "phase7.700.benchmark_report",
    "phase7.700.analysis_metadata",
    "phase1.110.raw_input_inventory",
    "phase1.110.primary_signature_handoff",
    "phase1.110.primary_perturbagen_coverage",
    "phase1.110.landmark_gene_handoff",
)

missing_input_ids = [
    artifact_id
    for artifact_id in INPUT_ARTIFACT_IDS
    if artifact_id not in artifact_registry["artifacts"]
]

if missing_input_ids:
    raise RuntimeError(
        "Notebook 701 is blocked until Notebook 700 registration passes. "
        "Missing artifacts: "
        + ", ".join(missing_input_ids)
    )

input_paths = {
    artifact_id: resolve_artifact_path(
        artifact_registry,
        artifact_id,
    )
    for artifact_id in INPUT_ARTIFACT_IDS
}



In [ ]:
# =============================================================================
# Load frozen Notebook 700 and LINCS handoffs
# =============================================================================

program_query_definitions = pd.read_csv(
    input_paths[
        "phase7.700.program_query_definitions"
    ],
    low_memory=False,
)

null_stratification_manifest = pd.read_csv(
    input_paths[
        "phase7.700.null_stratification_manifest"
    ],
    low_memory=False,
)

with input_paths[
    "phase7.700.benchmark_report"
].open("r", encoding="utf-8") as handle:
    benchmark_report = json.load(handle)

with input_paths[
    "phase7.700.analysis_metadata"
].open("r", encoding="utf-8") as handle:
    notebook700_metadata = json.load(handle)

raw_input_inventory = pd.read_csv(
    input_paths[
        "phase1.110.raw_input_inventory"
    ],
    low_memory=False,
)

primary_signature_handoff = pd.read_parquet(
    input_paths[
        "phase1.110.primary_signature_handoff"
    ]
)

primary_perturbagen_coverage = pd.read_csv(
    input_paths[
        "phase1.110.primary_perturbagen_coverage"
    ],
    low_memory=False,
)

landmark_gene_handoff = pd.read_csv(
    input_paths[
        "phase1.110.landmark_gene_handoff"
    ],
    low_memory=False,
)



In [ ]:
# =============================================================================
# Validate Notebook 700 handoff gate
# =============================================================================

if len(program_query_definitions) != 4485:
    raise RuntimeError("Unexpected Notebook 700 query-definition size.")

if len(null_stratification_manifest) != 10174:
    raise RuntimeError("Unexpected Notebook 700 null-manifest size.")

if len(primary_signature_handoff) != 362036:
    raise RuntimeError("Unexpected primary signature universe.")

if len(primary_perturbagen_coverage) != 7914:
    raise RuntimeError("Unexpected primary perturbagen universe.")

if (
    notebook700_metadata["sequential_monte_carlo"]["final_bmax"]
    != FINAL_BMAX
):
    raise RuntimeError("Notebook 700 Bmax does not match Notebook 701.")

if benchmark_report[
    "observed_program_perturbagen_connectivity_inspected"
] is not False:
    raise RuntimeError(
        "Notebook 700 benchmark is not marked outcome-blind."
    )



In [ ]:
# =============================================================================
# Resolve frozen raw resources and contract state
# =============================================================================

raw_resource_paths = (
    raw_input_inventory
    .set_index("resource")["relative_path"]
)

LINCS_LEVEL5_TRT_CP_PATH = (
    Paths.root
    / raw_resource_paths.loc["level5_trt_cp"]
)

if not LINCS_LEVEL5_TRT_CP_PATH.is_file():
    raise FileNotFoundError(
        f"Missing LINCS Level 5 GCTX: {LINCS_LEVEL5_TRT_CP_PATH}"
    )

contract_path = (
    Paths.root
    / "docs"
    / "contracts"
    / "phase7"
    / "PHASE7_ANALYSIS_CONTRACT.md"
)

contract_text = contract_path.read_text(
    encoding="utf-8"
)

CONTRACT_SHA256 = calculate_sha256(
    contract_path
)

if (
    "Bmax = 5,058,108"
    not in contract_text
):
    raise RuntimeError(
        "Phase 7 contract does not record the frozen final Bmax."
    )

if (
    "Generator.permutation"
    not in contract_text
):
    raise RuntimeError(
        "Phase 7 contract does not record the governing "
        "permutation implementation."
    )

HALLMARK_GMT_PATH = (
    Paths.msigdb
    / "h.all.v2026.1.Hs.symbols.gmt"
)

if not HALLMARK_GMT_PATH.is_file():
    raise FileNotFoundError(
        f"Missing frozen Hallmark GMT: {HALLMARK_GMT_PATH}"
    )

input_hashes = {
    artifact_id: artifact_registry[
        "artifacts"
    ][artifact_id]["sha256"]
    for artifact_id in INPUT_ARTIFACT_IDS
}

print("NOTEBOOK 701 INPUT GATE: PASS")
print("Registered inputs:", len(INPUT_ARTIFACT_IDS))
print("Primary signatures:", f"{len(primary_signature_handoff):,}")
print("Primary perturbagens:", f"{len(primary_perturbagen_coverage):,}")
print("Final Bmax:", f"{FINAL_BMAX:,}")


## Frozen Phase 7 query and null definitions

The following cells reconstruct only the prospectively frozen Notebook 700 definitions required for Notebook 701 execution. No observed connectivity is used to redefine the query, null strata, RNG mapping, or competitive-null matching plan.


In [ ]:
# =============================================================================
# Reconstruct frozen coupled program query
# =============================================================================

query_wide = (
    program_query_definitions[
        [
            "gene_id",
            "program_id",
            "frozen_weight",
        ]
    ]
    .pivot(
        index="gene_id",
        columns="program_id",
        values="frozen_weight",
    )
    .reindex(columns=list(PROGRAM_IDS))
    .reset_index()
)

if query_wide[list(PROGRAM_IDS)].isna().any().any():
    raise RuntimeError("Incomplete coupled three-program query weights.")

query_wide = (
    query_wide
    .merge(
        null_stratification_manifest[
            ["gene_id", "stratum_id"]
        ],
        on="gene_id",
        how="left",
        validate="one_to_one",
    )
    .sort_values(
        ["stratum_id", "gene_id"],
        kind="mergesort",
    )
    .reset_index(drop=True)
)

query_gene_ids = query_wide[
    "gene_id"
].to_numpy(dtype=np.int64)

frozen_weight_matrix = query_wide[
    list(PROGRAM_IDS)
].to_numpy(dtype=np.float64)

frozen_weight_norms = np.linalg.norm(
    frozen_weight_matrix,
    axis=0,
)

query_stratum_ids = query_wide[
    "stratum_id"
].to_numpy(dtype=np.int64)

STRATUM_IDS = tuple(
    sorted(
        np.unique(query_stratum_ids).tolist()
    )
)

query_stratum_row_indices = {
    int(stratum_id): np.flatnonzero(
        query_stratum_ids == int(stratum_id)
    ).astype(np.int64)
    for stratum_id in STRATUM_IDS
}

program_metadata = (
    program_query_definitions[
        [
            "program_id",
            "resistance_like_orientation_multiplier",
        ]
    ]
    .drop_duplicates()
    .set_index("program_id")
    .loc[list(PROGRAM_IDS)]
)

resistance_like_multiplier_vector = (
    program_metadata[
        "resistance_like_orientation_multiplier"
    ]
    .to_numpy(dtype=np.float64)
)

if not np.array_equal(
    resistance_like_multiplier_vector,
    np.array([-1.0, 1.0, 1.0]),
):
    raise RuntimeError(
        "Resistance-like orientation differs from frozen Phase 7 definition."
    )



In [ ]:
# =============================================================================
# Define deterministic Phase 7 RNG and permutation helpers
# =============================================================================

def make_phase7_rng(
    namespace,
    replicate_id,
    stratum_id,
):
    seed_sequence = np.random.SeedSequence(
        entropy=RNG_ENTROPY,
        spawn_key=(
            int(namespace),
            int(replicate_id),
            int(stratum_id),
        ),
    )

    return np.random.Generator(
        np.random.PCG64DXSM(seed_sequence)
    )


def phase7_uniform_permutation(
    n_positions,
    namespace,
    replicate_id,
    stratum_id,
):
    rng = make_phase7_rng(
        namespace=namespace,
        replicate_id=replicate_id,
        stratum_id=stratum_id,
    )

    return rng.permutation(
        int(n_positions)
    ).astype(
        np.int64,
        copy=False,
    )


def generate_coupled_pseudo_weights(
    replicate_id,
    namespace,
):
    pseudo_weights = frozen_weight_matrix.copy()

    for stratum_id in STRATUM_IDS:
        rows = query_stratum_row_indices[
            int(stratum_id)
        ]

        permutation = phase7_uniform_permutation(
            n_positions=len(rows),
            namespace=namespace,
            replicate_id=replicate_id,
            stratum_id=stratum_id,
        )

        pseudo_weights[
            rows,
            :,
        ] = frozen_weight_matrix[
            rows[permutation],
            :,
        ]

    return pseudo_weights




In [ ]:
# =============================================================================
# Freeze competitive-null matched blocks
# =============================================================================

# -------------------------------------------------------------------------
# Freeze competitive-null matched blocks before observed connectivity.
# -------------------------------------------------------------------------

null_gene_table = (
    null_stratification_manifest
    .sort_values(
        ["stratum_id", "gene_id"],
        kind="mergesort",
    )
    .reset_index(drop=True)
)

real_query_gene_set = set(
    query_gene_ids.tolist()
)

competitive_blocks = []

for stratum_id in STRATUM_IDS:
    stratum_id = int(stratum_id)

    demand_rows = query_stratum_row_indices[
        stratum_id
    ]

    demand = int(len(demand_rows))

    background = (
        null_gene_table.loc[
            null_gene_table["stratum_id"].eq(stratum_id)
            & ~null_gene_table["gene_id"].isin(
                real_query_gene_set
            )
        ]
    )

    if len(background) < demand:
        raise RuntimeError(
            "Competitive-null singleton stratum lacks sufficient "
            "background. A prospectively specified merge implementation "
            "would be required before observed connectivity is inspected."
        )

    competitive_blocks.append(
        {
            "stratum_ids": [stratum_id],
            "rng_stratum_id": stratum_id,
            "query_row_indices": (
                demand_rows.astype(int).tolist()
            ),
            "demand": demand,
            "eligible_background_gene_ids": (
                background["gene_id"]
                .astype(int)
                .sort_values()
                .tolist()
            ),
        }
    )

COMPETITIVE_PLAN_PATH = (
    Paths.perturbational_hypotheses
    / "701_competitive_matching_plan.json"
)

competitive_plan_payload = {
    "status": "FROZEN_BEFORE_OBSERVED_CONNECTIVITY",
    "scientific_results_inspected": False,
    "namespace": RNG_NAMESPACES[
        "competitive_null_sensitivity"
    ],
    "draws": 10000,
    "blocks": competitive_blocks,
}

COMPETITIVE_PLAN_HASH = hashlib.sha256(
    json.dumps(
        competitive_plan_payload,
        sort_keys=True,
        separators=(",", ":"),
    ).encode("utf-8")
).hexdigest()

if COMPETITIVE_PLAN_PATH.exists():
    with COMPETITIVE_PLAN_PATH.open(
        "r",
        encoding="utf-8",
    ) as handle:
        existing_competitive_plan = json.load(handle)

    if existing_competitive_plan != competitive_plan_payload:
        raise RuntimeError(
            "Non-identical competitive matching plan already exists."
        )
else:
    with COMPETITIVE_PLAN_PATH.open(
        "x",
        encoding="utf-8",
    ) as handle:
        json.dump(
            competitive_plan_payload,
            handle,
            indent=2,
            sort_keys=True,
        )
        handle.write("\n")

print("FROZEN QUERY / NULL DEFINITIONS: PASS")
print("Coupled query genes:", f"{len(query_gene_ids):,}")
print("Null strata:", len(STRATUM_IDS))
print("Competitive matched blocks:", len(competitive_blocks))
print("Competitive plan hash:", COMPETITIVE_PLAN_HASH)
print("Observed connectivity inspected: NO")


## Primary LINCS matrix and aggregation hierarchy

Notebook 701 reuses the Notebook 700 BING cache when its identity checks pass; otherwise it reconstructs the cache from the frozen GCTX input. The experimental hierarchy is then rebuilt deterministically from the frozen signature handoff.


In [ ]:
# =============================================================================
# Prepare primary BING cache identity
# =============================================================================

primary_signature_metadata = (
    primary_signature_handoff
    .sort_values(
        "gctx_signature_index",
        kind="mergesort",
    )
    .reset_index(drop=True)
)

primary_signature_indices = (
    primary_signature_metadata[
        "gctx_signature_index"
    ]
    .to_numpy(dtype=np.int64)
)

bing_gene_table = (
    null_stratification_manifest[
        [
            "gene_id",
            "gene_symbol",
            "feature_space",
        ]
    ]
    .drop_duplicates("gene_id")
    .sort_values(
        "gene_id",
        kind="mergesort",
    )
    .reset_index(drop=True)
)

n_bing_genes = len(bing_gene_table)
n_primary_signatures = len(
    primary_signature_metadata
)

TEMP_BING_MATRIX_PATH = (
    Path(tempfile.gettempdir())
    / "phase7_700_primary_bing_matrix.npy"
)

cache_expected_shape = (
    n_bing_genes,
    n_primary_signatures,
)

cache_reused = False



In [ ]:
# =============================================================================
# Reuse or reconstruct the primary BING expression cache
# =============================================================================

if TEMP_BING_MATRIX_PATH.is_file():
    try:
        candidate_cache = np.load(
            TEMP_BING_MATRIX_PATH,
            mmap_mode="r",
        )

        if (
            candidate_cache.shape
            == cache_expected_shape
            and candidate_cache.dtype
            == np.dtype(np.float32)
        ):
            primary_bing_matrix = (
                candidate_cache
            )
            cache_reused = True
    except Exception:
        cache_reused = False

if not cache_reused:
    projected_bytes = (
        n_bing_genes
        * n_primary_signatures
        * np.dtype(np.float32).itemsize
    )

    free_bytes = shutil.disk_usage(
        TEMP_BING_MATRIX_PATH.parent
    ).free

    if free_bytes < projected_bytes * 1.10:
        raise RuntimeError(
            "Insufficient temporary storage to reconstruct BING cache."
        )

    with h5py.File(
        LINCS_LEVEL5_TRT_CP_PATH,
        mode="r",
    ) as gctx:
        gctx_gene_ids = pd.to_numeric(
            pd.Series(
                [
                    (
                        value.decode("utf-8")
                        if isinstance(value, bytes)
                        else str(value)
                    )
                    for value in gctx[
                        "0/META/ROW/id"
                    ][:]
                ]
            ),
            errors="raise",
        ).astype("int64")

        gctx_gene_lookup = pd.Series(
            np.arange(
                len(gctx_gene_ids),
                dtype=np.int64,
            ),
            index=gctx_gene_ids.to_numpy(),
        )

        bing_gctx_indices = (
            bing_gene_table["gene_id"]
            .map(gctx_gene_lookup)
            .to_numpy(dtype=np.int64)
        )

        primary_bing_matrix = (
            np.lib.format.open_memmap(
                TEMP_BING_MATRIX_PATH,
                mode="w+",
                dtype=np.float32,
                shape=cache_expected_shape,
            )
        )

        matrix = gctx[
            "0/DATA/0/matrix"
        ]

        for block_start in range(
            0,
            matrix.shape[0],
            SIGNATURE_BLOCK_SIZE,
        ):
            block_stop = min(
                block_start
                + SIGNATURE_BLOCK_SIZE,
                matrix.shape[0],
            )

            left = np.searchsorted(
                primary_signature_indices,
                block_start,
                side="left",
            )

            right = np.searchsorted(
                primary_signature_indices,
                block_stop,
                side="left",
            )

            if left == right:
                continue

            block = matrix[
                block_start:block_stop,
                :,
            ]

            local_rows = (
                primary_signature_indices[
                    left:right
                ]
                - block_start
            )

            primary_bing_matrix[
                :,
                left:right,
            ] = block[
                local_rows
            ][
                :,
                bing_gctx_indices
            ].T

        primary_bing_matrix.flush()

    primary_bing_matrix = np.load(
        TEMP_BING_MATRIX_PATH,
        mmap_mode="r",
    )


bing_row_lookup = pd.Series(
    np.arange(
        len(bing_gene_table),
        dtype=np.int64,
    ),
    index=bing_gene_table[
        "gene_id"
    ].to_numpy(dtype=np.int64),
)

query_bing_row_indices = (
    pd.Series(query_gene_ids)
    .map(bing_row_lookup)
)

if query_bing_row_indices.isna().any():
    raise RuntimeError(
        "Frozen query gene absent from BING cache."
    )

query_bing_row_indices = (
    query_bing_row_indices
    .to_numpy(dtype=np.int64)
)



In [ ]:
# =============================================================================
# Validate the BING cache against raw GCTX
# =============================================================================

# Independent sentinel validation against raw GCTX.
with h5py.File(
    LINCS_LEVEL5_TRT_CP_PATH,
    mode="r",
) as gctx:
    matrix = gctx["0/DATA/0/matrix"]

    sentinel_bing_rows = np.unique(
        np.linspace(
            0,
            len(bing_gene_table) - 1,
            11,
            dtype=np.int64,
        )
    )

    sentinel_signature_positions = np.unique(
        np.linspace(
            0,
            len(primary_signature_indices) - 1,
            11,
            dtype=np.int64,
        )
    )

    for bing_row in sentinel_bing_rows:
        gene_id = int(
            bing_gene_table.iloc[
                bing_row
            ]["gene_id"]
        )

        raw_gene_index = int(
            null_stratification_manifest.loc[
                null_stratification_manifest[
                    "gene_id"
                ].eq(gene_id),
                "gctx_gene_index",
            ].iloc[0]
        )

        for signature_position in (
            sentinel_signature_positions
        ):
            raw_signature_index = int(
                primary_signature_indices[
                    signature_position
                ]
            )

            cached_value = float(
                primary_bing_matrix[
                    bing_row,
                    signature_position,
                ]
            )

            raw_value = float(
                matrix[
                    raw_signature_index,
                    raw_gene_index,
                ]
            )

            if cached_value != raw_value:
                raise RuntimeError(
                    "BING cache sentinel validation failed."
                )

print("PRIMARY BING CACHE: VALIDATED")
print("Cache action:", "REUSED" if cache_reused else "RECONSTRUCTED")
print("Shape:", primary_bing_matrix.shape)
print("Dtype:", primary_bing_matrix.dtype)
print("Path:", TEMP_BING_MATRIX_PATH)
print(
    "Raw GCTX sentinel comparisons:",
    int(
        len(sentinel_bing_rows)
        * len(sentinel_signature_positions)
    ),
)


In [ ]:
# =============================================================================
# Define frozen hierarchy keys and mapping helpers
# =============================================================================

condition_keys = [
    "pert_id",
    "cell_iname",
    "cell_lineage",
    "pert_dose",
    "pert_dose_unit",
    "pert_time",
    "pert_time_unit",
]

timepoint_keys = [
    "pert_id",
    "cell_iname",
    "cell_lineage",
    "pert_time",
    "pert_time_unit",
]

cell_line_keys = [
    "pert_id",
    "cell_iname",
    "cell_lineage",
]

lineage_keys = [
    "pert_id",
    "cell_lineage",
]


def factorize_group_keys(
    frame,
    columns,
):
    keys = pd.MultiIndex.from_frame(
        frame[columns]
    )

    codes, uniques = pd.factorize(
        keys,
        sort=True,
    )

    if isinstance(
        uniques,
        pd.MultiIndex,
    ):
        uniques = uniques.set_names(
            columns
        )

    return (
        codes.astype(np.int32),
        uniques,
    )


def build_parent_map(
    child_ids,
    parent_ids,
    n_children,
):
    mapping = (
        pd.DataFrame(
            {
                "child_id": child_ids,
                "parent_id": parent_ids,
            }
        )
        .drop_duplicates()
        .sort_values(
            "child_id",
            kind="mergesort",
        )
        .reset_index(drop=True)
    )

    if mapping["child_id"].duplicated().any():
        raise RuntimeError(
            "Hierarchy child maps to multiple parents."
        )

    if not np.array_equal(
        mapping["child_id"].to_numpy(),
        np.arange(
            n_children,
            dtype=np.int64,
        ),
    ):
        raise RuntimeError(
            "Hierarchy IDs are not complete and contiguous."
        )

    return mapping[
        "parent_id"
    ].to_numpy(dtype=np.int32)




In [ ]:
# =============================================================================
# Construct deterministic hierarchy indices
# =============================================================================

condition_id, condition_index = factorize_group_keys(
    primary_signature_metadata,
    condition_keys,
)

timepoint_id, timepoint_index = factorize_group_keys(
    primary_signature_metadata,
    timepoint_keys,
)

cell_line_id, cell_line_index = factorize_group_keys(
    primary_signature_metadata,
    cell_line_keys,
)

lineage_id, lineage_index = factorize_group_keys(
    primary_signature_metadata,
    lineage_keys,
)

perturbagen_id, perturbagen_index = factorize_group_keys(
    primary_signature_metadata,
    ["pert_id"],
)

condition_to_timepoint = build_parent_map(
    condition_id,
    timepoint_id,
    len(condition_index),
)

timepoint_to_cell_line = build_parent_map(
    timepoint_id,
    cell_line_id,
    len(timepoint_index),
)

cell_line_to_lineage = build_parent_map(
    cell_line_id,
    lineage_id,
    len(cell_line_index),
)

lineage_to_perturbagen = build_parent_map(
    lineage_id,
    perturbagen_id,
    len(lineage_index),
)

condition_to_perturbagen = build_parent_map(
    condition_id,
    perturbagen_id,
    len(condition_index),
)

timepoint_to_perturbagen = build_parent_map(
    timepoint_id,
    perturbagen_id,
    len(timepoint_index),
)

cell_line_to_perturbagen_direct = build_parent_map(
    cell_line_id,
    perturbagen_id,
    len(cell_line_index),
)




In [ ]:
# =============================================================================
# Define hierarchical median-reduction helpers
# =============================================================================

def build_median_reduction_plan(
    parent_ids,
):
    parent_ids = np.asarray(
        parent_ids,
        dtype=np.int64,
    )

    order = np.argsort(
        parent_ids,
        kind="stable",
    )

    sorted_parent_ids = parent_ids[
        order
    ]

    unique_parents, starts, counts = np.unique(
        sorted_parent_ids,
        return_index=True,
        return_counts=True,
    )

    if not np.array_equal(
        unique_parents,
        np.arange(
            len(unique_parents),
            dtype=np.int64,
        ),
    ):
        raise RuntimeError(
            "Parent IDs are not complete and contiguous."
        )

    plan = []

    for group_size in np.unique(
        counts
    ):
        mask = counts == group_size

        plan.append(
            {
                "group_size": int(group_size),
                "parent_ids": unique_parents[
                    mask
                ].astype(np.int32),
                "child_indices": order[
                    starts[mask, np.newaxis]
                    + np.arange(
                        group_size,
                        dtype=np.int64,
                    )
                ].astype(np.int32),
            }
        )

    return plan, len(unique_parents)


def apply_median_reduction(
    values,
    plan,
    n_parents,
):
    values = np.asarray(
        values,
        dtype=np.float64,
    )

    was_1d = values.ndim == 1

    if was_1d:
        values = values[
            :,
            np.newaxis,
        ]

    reduced = np.empty(
        (
            n_parents,
            values.shape[1],
        ),
        dtype=np.float64,
    )

    for block in plan:
        reduced[
            block["parent_ids"],
            :,
        ] = np.median(
            values[
                block["child_indices"],
                :,
            ],
            axis=1,
        )

    return (
        reduced[:, 0]
        if was_1d
        else reduced
    )




In [ ]:
# =============================================================================
# Build and validate frozen aggregation plans
# =============================================================================

aggregation_plans = {
    "signature_to_condition": build_median_reduction_plan(
        condition_id
    ),
    "condition_to_timepoint": build_median_reduction_plan(
        condition_to_timepoint
    ),
    "timepoint_to_cell_line": build_median_reduction_plan(
        timepoint_to_cell_line
    ),
    "cell_line_to_lineage": build_median_reduction_plan(
        cell_line_to_lineage
    ),
    "lineage_to_perturbagen": build_median_reduction_plan(
        lineage_to_perturbagen
    ),
}


def aggregate_signature_scores(
    signature_scores,
):
    values = np.asarray(
        signature_scores,
        dtype=np.float64,
    )

    condition = apply_median_reduction(
        values,
        *aggregation_plans[
            "signature_to_condition"
        ],
    )

    timepoint = apply_median_reduction(
        condition,
        *aggregation_plans[
            "condition_to_timepoint"
        ],
    )

    cell_line = apply_median_reduction(
        timepoint,
        *aggregation_plans[
            "timepoint_to_cell_line"
        ],
    )

    lineage = apply_median_reduction(
        cell_line,
        *aggregation_plans[
            "cell_line_to_lineage"
        ],
    )

    perturbagen = apply_median_reduction(
        lineage,
        *aggregation_plans[
            "lineage_to_perturbagen"
        ],
    )

    return {
        "condition": condition,
        "timepoint": timepoint,
        "cell_line": cell_line,
        "lineage": lineage,
        "perturbagen": perturbagen,
    }


level_to_perturbagen = {
    "condition": condition_to_perturbagen,
    "timepoint": timepoint_to_perturbagen,
    "cell_line": cell_line_to_perturbagen_direct,
    "lineage": lineage_to_perturbagen,
    "perturbagen": np.arange(
        len(perturbagen_index),
        dtype=np.int32,
    ),
}

signature_indices_by_perturbagen = [
    np.flatnonzero(
        perturbagen_id == pert_index
    ).astype(np.int64)
    for pert_index in range(
        len(perturbagen_index)
    )
]

expected_counts = {
    "signatures": 362036,
    "conditions": 325190,
    "timepoints": 113097,
    "cell_lines": 90673,
    "lineages": 55827,
    "perturbagens": 7914,
}

observed_counts = {
    "signatures": len(primary_signature_metadata),
    "conditions": len(condition_index),
    "timepoints": len(timepoint_index),
    "cell_lines": len(cell_line_index),
    "lineages": len(lineage_index),
    "perturbagens": len(perturbagen_index),
}

if observed_counts != expected_counts:
    raise RuntimeError(
        f"Frozen hierarchy mismatch: {observed_counts}"
    )

print("FROZEN HIERARCHY: VALIDATED")
print(observed_counts)


## Primary execution helpers

These cells define and validate the exact observed-scoring, batched-null, active-set aggregation, checkpoint/restart, and multiplicity machinery before any long-running primary Monte Carlo execution begins.


In [ ]:
# =============================================================================
# Define observed primary cosine scoring
# =============================================================================

def score_observed_primary():
    n_signatures = len(
        primary_signature_metadata
    )

    signature_raw = np.empty(
        (
            n_signatures,
            len(PROGRAM_IDS),
        ),
        dtype=np.float64,
    )

    signature_norms = np.empty(
        n_signatures,
        dtype=np.float64,
    )

    for start in range(
        0,
        n_signatures,
        SIGNATURE_BLOCK_SIZE,
    ):
        stop = min(
            start + SIGNATURE_BLOCK_SIZE,
            n_signatures,
        )

        block = np.asarray(
            primary_bing_matrix[
                query_bing_row_indices,
                start:stop,
            ],
            dtype=np.float64,
        )

        norms = np.linalg.norm(
            block,
            axis=0,
        )

        if not np.isfinite(block).all():
            raise RuntimeError(
                "Primary BING block contains non-finite values."
            )

        if not np.all(norms > 0):
            raise RuntimeError(
                "Primary query-support vector has zero norm."
            )

        numerators = (
            frozen_weight_matrix.T
            @ block
        )

        scores = (
            numerators
            / (
                frozen_weight_norms[
                    :,
                    np.newaxis,
                ]
                * norms[
                    np.newaxis,
                    :
                ]
            )
        ).T

        if np.max(
            np.abs(scores)
        ) > 1 + NUMERICAL_ATOL:
            raise RuntimeError(
                "Primary cosine exceeded theoretical range."
            )

        signature_raw[
            start:stop,
            :,
        ] = scores

        signature_norms[
            start:stop
        ] = norms

    return (
        signature_raw,
        signature_norms,
    )




In [ ]:
# =============================================================================
# Define batched null-scoring helpers
# =============================================================================

def generate_pseudo_weight_batch(
    replicate_ids,
    namespace,
):
    return np.stack(
        [
            generate_coupled_pseudo_weights(
                replicate_id=int(
                    replicate_id
                ),
                namespace=namespace,
            )
            for replicate_id in (
                replicate_ids
            )
        ],
        axis=0,
    )


def score_signature_block_batch(
    signature_block,
    signature_norms,
    weight_tensor,
):
    signature_block = np.asarray(
        signature_block,
        dtype=np.float64,
    )

    signature_norms = np.asarray(
        signature_norms,
        dtype=np.float64,
    )

    weight_tensor = np.asarray(
        weight_tensor,
        dtype=np.float64,
    )

    n_replicates = (
        weight_tensor.shape[0]
    )

    flat_weights = (
        weight_tensor
        .transpose(1, 0, 2)
        .reshape(
            weight_tensor.shape[1],
            n_replicates
            * len(PROGRAM_IDS),
        )
    )

    numerators = (
        flat_weights.T
        @ signature_block
    )

    repeated_weight_norms = np.tile(
        frozen_weight_norms,
        n_replicates,
    )

    scores = (
        numerators
        / (
            repeated_weight_norms[
                :,
                np.newaxis,
            ]
            * signature_norms[
                np.newaxis,
                :
            ]
        )
    )

    return (
        scores.T
        .reshape(
            signature_block.shape[1],
            n_replicates,
            len(PROGRAM_IDS),
        )
    )




In [ ]:
# =============================================================================
# Define active median-reduction helper
# =============================================================================

def apply_active_union_reduction(
    values,
    plan,
    n_parents,
    parent_to_perturbagen,
    active_perturbagen_union,
):
    values = np.asarray(
        values,
        dtype=np.float64,
    )

    reduced = np.full(
        (
            n_parents,
            values.shape[1],
        ),
        np.nan,
        dtype=np.float64,
    )

    for block in plan:
        parent_ids = block[
            "parent_ids"
        ].astype(
            np.int64,
            copy=False,
        )

        active_parent = (
            active_perturbagen_union[
                parent_to_perturbagen[
                    parent_ids
                ]
            ]
        )

        if not active_parent.any():
            continue

        selected_parent_ids = (
            parent_ids[
                active_parent
            ]
        )

        selected_child_indices = (
            block[
                "child_indices"
            ][
                active_parent,
                :
            ]
        )

        selected_values = values[
            selected_child_indices,
            :,
        ]

        if not np.isfinite(
            selected_values
        ).all():
            raise RuntimeError(
                "Active hierarchical aggregation encountered "
                "non-finite child values."
            )

        reduced[
            selected_parent_ids,
            :,
        ] = np.median(
            selected_values,
            axis=1,
        )

    return reduced




In [ ]:
# =============================================================================
# Define active batched aggregation to perturbagen level
# =============================================================================

def aggregate_active_batch_to_perturbagen(
    active_signature_scores,
    active_signature_indices,
    active_perturbagen_union,
):
    active_signature_scores = np.asarray(
        active_signature_scores,
        dtype=np.float64,
    )

    n_columns = (
        active_signature_scores.shape[1]
    )

    values = np.full(
        (
            len(primary_signature_metadata),
            n_columns,
        ),
        np.nan,
        dtype=np.float64,
    )

    values[
        active_signature_indices,
        :,
    ] = active_signature_scores

    values = apply_active_union_reduction(
        values,
        *aggregation_plans[
            "signature_to_condition"
        ],
        parent_to_perturbagen=(
            condition_to_perturbagen
        ),
        active_perturbagen_union=(
            active_perturbagen_union
        ),
    )

    values = apply_active_union_reduction(
        values,
        *aggregation_plans[
            "condition_to_timepoint"
        ],
        parent_to_perturbagen=(
            timepoint_to_perturbagen
        ),
        active_perturbagen_union=(
            active_perturbagen_union
        ),
    )

    values = apply_active_union_reduction(
        values,
        *aggregation_plans[
            "timepoint_to_cell_line"
        ],
        parent_to_perturbagen=(
            cell_line_to_perturbagen_direct
        ),
        active_perturbagen_union=(
            active_perturbagen_union
        ),
    )

    values = apply_active_union_reduction(
        values,
        *aggregation_plans[
            "cell_line_to_lineage"
        ],
        parent_to_perturbagen=(
            lineage_to_perturbagen
        ),
        active_perturbagen_union=(
            active_perturbagen_union
        ),
    )

    values = apply_active_union_reduction(
        values,
        *aggregation_plans[
            "lineage_to_perturbagen"
        ],
        parent_to_perturbagen=np.arange(
            len(perturbagen_index),
            dtype=np.int32,
        ),
        active_perturbagen_union=(
            active_perturbagen_union
        ),
    )

    return values




In [ ]:
# =============================================================================
# Freeze primary execution specification and checkpoint schema
# =============================================================================

EXECUTION_SPEC_PAYLOAD = {
    "contract_sha256": CONTRACT_SHA256,
    "program_order": list(PROGRAM_IDS),
    "query_artifact_sha256": input_hashes[
        "phase7.700.program_query_definitions"
    ],
    "null_manifest_sha256": input_hashes[
        "phase7.700.null_stratification_manifest"
    ],
    "rng_entropy": RNG_ENTROPY,
    "rng_namespace": RNG_NAMESPACES[
        "primary_conditional_null"
    ],
    "rng_bit_generator": "PCG64DXSM",
    "permutation_mapping": (
        "NumPy Generator.permutation"
    ),
    "numpy_version": np.__version__,
    "h": MC_EXCEEDANCE_TARGET,
    "Bmax": FINAL_BMAX,
    "batch_size": MC_BATCH_SIZE,
    "checkpoint_interval": CHECKPOINT_INTERVAL,
}

EXECUTION_SPEC_HASH = hashlib.sha256(
    json.dumps(
        EXECUTION_SPEC_PAYLOAD,
        sort_keys=True,
        separators=(",", ":"),
    ).encode("utf-8")
).hexdigest()

CHECKPOINT_SCHEMA_VERSION = (
    "phase7_701_primary_mc_v2"
)

PRIMARY_CHECKPOINT_PATH = (
    Paths.perturbational_hypotheses
    / "701_primary_mc_checkpoint.npz"
)




In [ ]:
# =============================================================================
# Define primary checkpoint read/write helpers
# =============================================================================

def atomic_write_primary_checkpoint(
    path,
    next_replicate_id,
    mc_g,
    mc_L,
    active_mask,
    stop_code,
    family_hash,
    observed_hash,
):
    path = Path(path)

    temp_path = path.with_suffix(
        path.suffix + ".tmp"
    )

    with temp_path.open(
        "wb"
    ) as handle:
        np.savez(
            handle,
            schema_version=np.array(
                [CHECKPOINT_SCHEMA_VERSION]
            ),
            final_bmax=np.array(
                [FINAL_BMAX],
                dtype=np.int64,
            ),
            h=np.array(
                [MC_EXCEEDANCE_TARGET],
                dtype=np.int64,
            ),
            next_replicate_id=np.array(
                [next_replicate_id],
                dtype=np.int64,
            ),
            mc_g=np.asarray(
                mc_g,
                dtype=np.int32,
            ),
            mc_L=np.asarray(
                mc_L,
                dtype=np.int32,
            ),
            active_mask=np.asarray(
                active_mask,
                dtype=bool,
            ),
            stop_code=np.asarray(
                stop_code,
                dtype=np.int8,
            ),
            family_hash=np.array(
                [family_hash]
            ),
            observed_hash=np.array(
                [observed_hash]
            ),
            execution_spec_hash=np.array(
                [EXECUTION_SPEC_HASH]
            ),
        )

        handle.flush()
        os.fsync(
            handle.fileno()
        )

    os.replace(
        temp_path,
        path,
    )


def read_primary_checkpoint(
    path,
    family_hash,
    observed_hash,
):
    with np.load(
        path,
        allow_pickle=False,
    ) as checkpoint:
        if str(
            checkpoint[
                "schema_version"
            ][0]
        ) != CHECKPOINT_SCHEMA_VERSION:
            raise RuntimeError(
                "Unexpected primary checkpoint schema."
            )

        if int(
            checkpoint[
                "final_bmax"
            ][0]
        ) != FINAL_BMAX:
            raise RuntimeError(
                "Checkpoint Bmax mismatch."
            )

        if int(
            checkpoint["h"][0]
        ) != MC_EXCEEDANCE_TARGET:
            raise RuntimeError(
                "Checkpoint h mismatch."
            )

        if str(
            checkpoint[
                "family_hash"
            ][0]
        ) != family_hash:
            raise RuntimeError(
                "Checkpoint family hash mismatch."
            )

        if str(
            checkpoint[
                "observed_hash"
            ][0]
        ) != observed_hash:
            raise RuntimeError(
                "Checkpoint observed-score hash mismatch."
            )

        if str(
            checkpoint[
                "execution_spec_hash"
            ][0]
        ) != EXECUTION_SPEC_HASH:
            raise RuntimeError(
                "Checkpoint execution-spec hash mismatch."
            )

        return {
            "next_replicate_id": int(
                checkpoint[
                    "next_replicate_id"
                ][0]
            ),
            "mc_g": checkpoint[
                "mc_g"
            ].copy(),
            "mc_L": checkpoint[
                "mc_L"
            ].copy(),
            "active_mask": checkpoint[
                "active_mask"
            ].copy(),
            "stop_code": checkpoint[
                "stop_code"
            ].copy(),
        }




In [ ]:
# =============================================================================
# Define global BH and BY adjustment helpers
# =============================================================================

def adjust_bh(
    p_values,
):
    p_values = np.asarray(
        p_values,
        dtype=np.float64,
    )

    m = len(p_values)

    order = np.argsort(
        p_values,
        kind="mergesort",
    )

    ranked = p_values[
        order
    ]

    adjusted_ranked = (
        ranked
        * m
        / np.arange(
            1,
            m + 1,
            dtype=np.float64,
        )
    )

    adjusted_ranked = np.minimum.accumulate(
        adjusted_ranked[::-1]
    )[::-1]

    adjusted = np.empty(
        m,
        dtype=np.float64,
    )

    adjusted[
        order
    ] = np.minimum(
        adjusted_ranked,
        1.0,
    )

    return adjusted


def adjust_by(
    p_values,
):
    m = len(p_values)

    harmonic = np.sum(
        1.0
        / np.arange(
            1,
            m + 1,
            dtype=np.float64,
        )
    )

    return np.minimum(
        adjust_bh(p_values)
        * harmonic,
        1.0,
    )




In [ ]:
# =============================================================================
# Validate multiplicity helpers
# =============================================================================

# Independent multiplicity reference on a toy vector.
toy_p = np.array(
    [0.01, 0.04, 0.03, 0.002],
    dtype=np.float64,
)

np.testing.assert_allclose(
    adjust_bh(toy_p),
    np.array(
        [0.02, 0.04, 0.04, 0.008],
        dtype=np.float64,
    ),
    rtol=0,
    atol=1e-15,
)

# -------------------------------------------------------------------------


In [ ]:
# =============================================================================
# Validate batched active hierarchy against full reference
# =============================================================================

# Exact synthetic validation of batched active aggregation.
# -------------------------------------------------------------------------

synthetic_columns = np.column_stack(
    [
        np.sin(
            np.arange(
                len(primary_signature_metadata),
                dtype=np.float64,
            )
            / divisor
        )
        + np.cos(
            np.arange(
                len(primary_signature_metadata),
                dtype=np.float64,
            )
            / (
                divisor + 3.0
            )
        )
        for divisor in (
            17.0,
            23.0,
            31.0,
            41.0,
            53.0,
            67.0,
        )
    ]
)

synthetic_full_reference = (
    aggregate_signature_scores(
        synthetic_columns
    )[
        "perturbagen"
    ]
)

validation_perturbagens = np.arange(
    50,
    dtype=np.int64,
)

validation_union = np.zeros(
    len(perturbagen_ids)
    if "perturbagen_ids" in globals()
    else len(perturbagen_index),
    dtype=bool,
)

validation_union[
    validation_perturbagens
] = True

validation_signature_indices = (
    np.concatenate(
        [
            signature_indices_by_perturbagen[
                int(pert_index)
            ]
            for pert_index in (
                validation_perturbagens
            )
        ]
    )
    .astype(
        np.int64,
        copy=False,
    )
)

synthetic_batched_observed = (
    aggregate_active_batch_to_perturbagen(
        active_signature_scores=(
            synthetic_columns[
                validation_signature_indices,
                :,
            ]
        ),
        active_signature_indices=(
            validation_signature_indices
        ),
        active_perturbagen_union=(
            validation_union
        ),
    )
)

np.testing.assert_allclose(
    synthetic_batched_observed[
        validation_union,
        :,
    ],
    synthetic_full_reference[
        validation_union,
        :,
    ],
    rtol=NUMERICAL_RTOL,
    atol=NUMERICAL_ATOL,
)

if not np.isnan(
    synthetic_batched_observed[
        ~validation_union,
        :,
    ]
).all():
    raise RuntimeError(
        "Inactive perturbagens were not retained as NaN "
        "in batched aggregation validation."
    )

del synthetic_columns
del synthetic_full_reference
del synthetic_batched_observed

print(
    "Batched active hierarchical aggregation reference validation: PASS"
)

print("PRIMARY EXECUTION FUNCTIONS: VALIDATED")
print("Checkpoint:", project_relative_path(PRIMARY_CHECKPOINT_PATH))
print("Checkpoint interval:", CHECKPOINT_INTERVAL)
print("Execution-spec hash:", EXECUTION_SPEC_HASH)


## Observed primary connectivity

Observed weighted-cosine connectivity is now computed under the fully frozen Phase 7 specification. The cells below materialize lineage context, coverage/evaluability metadata, and the complete 23,742-hypothesis primary family before sequential Monte Carlo inference.


In [ ]:
# =============================================================================
# Compute observed signature and hierarchical connectivity
# =============================================================================

observed_signature_raw, query_signature_norms = (
    score_observed_primary()
)

observed_raw_aggregation = (
    aggregate_signature_scores(
        observed_signature_raw
    )
)

observed_rl_aggregation = {
    level: (
        values
        * resistance_like_multiplier_vector[
            np.newaxis,
            :
        ]
    )
    for level, values in (
        observed_raw_aggregation.items()
    )
}

perturbagen_ids = np.array(
    sorted(
        primary_signature_metadata[
            "pert_id"
        ].unique()
    ),
    dtype=object,
)

if len(perturbagen_ids) != 7914:
    raise RuntimeError(
        "Unexpected perturbagen family size."
    )

# Factorization with sort=True must reproduce the same perturbagen order.
perturbagen_index_values = np.array(
    perturbagen_index.get_level_values(
        0
    ).tolist(),
    dtype=object,
)

if not np.array_equal(
    perturbagen_ids,
    perturbagen_index_values,
):
    raise RuntimeError(
        "Perturbagen index order mismatch."
    )




In [ ]:
# =============================================================================
# Materialize lineage-level connectivity
# =============================================================================

# -------------------------------------------------------------------------
# Lineage-level stable table.
# -------------------------------------------------------------------------

lineage_index_frame = (
    lineage_index
    .to_frame(index=False)
)

lineage_index_frame.columns = (
    lineage_keys
)

lineage_cell_counts = (
    pd.DataFrame(
        {
            "lineage_id": (
                cell_line_to_lineage
            ),
            "cell_line_id": np.arange(
                len(cell_line_index),
                dtype=np.int64,
            ),
        }
    )
    .groupby(
        "lineage_id",
        as_index=False,
    )
    .agg(
        n_evaluable_cell_lines=(
            "cell_line_id",
            "size",
        )
    )
)

lineage_rows = []

for program_index, program_id in enumerate(
    PROGRAM_IDS
):
    block = (
        lineage_index_frame
        .copy()
        .assign(
            program_id=program_id,
            resistance_like_orientation_multiplier=int(
                resistance_like_multiplier_vector[
                    program_index
                ]
            ),
            C_raw=observed_raw_aggregation[
                "lineage"
            ][:, program_index],
            C_RL=observed_rl_aggregation[
                "lineage"
            ][:, program_index],
        )
        .reset_index(
            names="lineage_id"
        )
        .merge(
            lineage_cell_counts,
            on="lineage_id",
            how="left",
            validate="one_to_one",
        )
    )

    lineage_rows.append(block)

lineage_connectivity = (
    pd.concat(
        lineage_rows,
        ignore_index=True,
    )
    .sort_values(
        [
            "program_id",
            "pert_id",
            "cell_lineage",
        ],
        kind="mergesort",
    )
    .reset_index(drop=True)
)




In [ ]:
# =============================================================================
# Materialize perturbagen coverage and lineage heterogeneity
# =============================================================================

# -------------------------------------------------------------------------
# Coverage/evaluability counts.
# -------------------------------------------------------------------------

signature_counts = (
    primary_signature_metadata
    .groupby(
        "pert_id",
        as_index=False,
    )
    .agg(
        n_evaluable_signatures=(
            "sig_id",
            "size",
        ),
        n_evaluable_cell_lines=(
            "cell_iname",
            "nunique",
        ),
        n_evaluable_lineages=(
            "cell_lineage",
            "nunique",
        ),
    )
)

condition_counts = (
    primary_signature_metadata[
        condition_keys
    ]
    .drop_duplicates()
    .groupby(
        "pert_id",
        as_index=False,
    )
    .size()
    .rename(
        columns={
            "size": "n_evaluable_conditions",
        }
    )
)

timepoint_counts = (
    primary_signature_metadata[
        timepoint_keys
    ]
    .drop_duplicates()
    .groupby(
        "pert_id",
        as_index=False,
    )
    .size()
    .rename(
        columns={
            "size": "n_evaluable_timepoints",
        }
    )
)

coverage_counts = (
    signature_counts
    .merge(
        condition_counts,
        on="pert_id",
        how="left",
        validate="one_to_one",
    )
    .merge(
        timepoint_counts,
        on="pert_id",
        how="left",
        validate="one_to_one",
    )
)

lineage_summary = (
    lineage_connectivity
    .groupby(
        [
            "program_id",
            "pert_id",
        ],
        as_index=False,
    )
    .agg(
        lineage_q25_C_RL=(
            "C_RL",
            lambda values: float(
                np.quantile(
                    values.to_numpy(
                        dtype=np.float64
                    ),
                    0.25,
                )
            ),
        ),
        lineage_q75_C_RL=(
            "C_RL",
            lambda values: float(
                np.quantile(
                    values.to_numpy(
                        dtype=np.float64
                    ),
                    0.75,
                )
            ),
        ),
        fraction_negative_lineages=(
            "C_RL",
            lambda values: float(
                np.mean(
                    values.to_numpy(
                        dtype=np.float64
                    ) < 0
                )
            ),
        ),
    )
)

lineage_summary[
    "lineage_iqr_C_RL"
] = (
    lineage_summary[
        "lineage_q75_C_RL"
    ]
    - lineage_summary[
        "lineage_q25_C_RL"
    ]
)




In [ ]:
# =============================================================================
# Freeze complete primary hypothesis family
# =============================================================================

# -------------------------------------------------------------------------
# Complete frozen 23,742 hypothesis family.
# -------------------------------------------------------------------------

primary_family_rows = []

for program_index, program_id in enumerate(
    PROGRAM_IDS
):
    block = pd.DataFrame(
        {
            "program_id": program_id,
            "program_index": program_index,
            "pert_id": perturbagen_ids,
            "perturbagen_index": np.arange(
                len(perturbagen_ids),
                dtype=np.int64,
            ),
            "resistance_like_orientation_multiplier": int(
                resistance_like_multiplier_vector[
                    program_index
                ]
            ),
            "C_raw": observed_raw_aggregation[
                "perturbagen"
            ][:, program_index],
            "C_RL": observed_rl_aggregation[
                "perturbagen"
            ][:, program_index],
        }
    )

    primary_family_rows.append(block)

primary_family = (
    pd.concat(
        primary_family_rows,
        ignore_index=True,
    )
    .merge(
        coverage_counts,
        on="pert_id",
        how="left",
        validate="many_to_one",
    )
    .merge(
        lineage_summary,
        on=[
            "program_id",
            "pert_id",
        ],
        how="left",
        validate="one_to_one",
    )
    .sort_values(
        [
            "program_index",
            "perturbagen_index",
        ],
        kind="mergesort",
    )
    .reset_index(drop=True)
)

primary_family[
    "evaluability_status"
] = np.where(
    (
        primary_family[
            "n_evaluable_cell_lines"
        ]
        >= PRIMARY_MIN_CELL_LINES
    )
    & (
        primary_family[
            "n_evaluable_lineages"
        ]
        >= PRIMARY_MIN_LINEAGES
    ),
    "EVALUABLE",
    "NOT_EVALUABLE",
)

primary_family[
    "directional_status"
] = np.where(
    primary_family[
        "evaluability_status"
    ].eq("EVALUABLE")
    & primary_family[
        "C_RL"
    ].lt(0),
    "INVERSE",
    np.where(
        primary_family[
            "evaluability_status"
        ].eq("NOT_EVALUABLE"),
        "NOT_EVALUABLE",
        "NON_INVERSE",
    ),
)

active_primary_mask_2d = (
    primary_family[
        "directional_status"
    ]
    .eq("INVERSE")
    .to_numpy()
    .reshape(
        len(PROGRAM_IDS),
        len(perturbagen_ids),
    )
    .T
)

family_key_text = "\n".join(
    primary_family[
        ["program_id", "pert_id"]
    ]
    .astype(str)
    .agg("\t".join, axis=1)
    .tolist()
)

PRIMARY_FAMILY_HASH = hashlib.sha256(
    family_key_text.encode("utf-8")
).hexdigest()

observed_hash_payload = np.ascontiguousarray(
    primary_family[
        "C_RL"
    ].to_numpy(dtype=np.float64)
).tobytes()

OBSERVED_PRIMARY_HASH = hashlib.sha256(
    observed_hash_payload
).hexdigest()

if len(primary_family) != PRIMARY_FAMILY_SIZE:
    raise RuntimeError(
        "Primary hypothesis family is not 23,742."
    )

if not np.isfinite(
    primary_family[
        ["C_raw", "C_RL"]
    ].to_numpy(dtype=np.float64)
).all():
    raise RuntimeError(
        "Observed primary connectivity contains non-finite values."
    )

print("OBSERVED PRIMARY CONNECTIVITY: COMPUTED")
print("Primary hypotheses:", f"{len(primary_family):,}")
print(
    "Inverse/evaluable hypotheses entering MC:",
    f"{int(active_primary_mask_2d.sum()):,}",
)
print(
    "Non-inverse hypotheses:",
    f"{primary_family['directional_status'].eq('NON_INVERSE').sum():,}",
)
print(
    "Not-evaluable hypotheses:",
    f"{primary_family['evaluability_status'].eq('NOT_EVALUABLE').sum():,}",
)
print("Primary family hash:", PRIMARY_FAMILY_HASH)
print("Observed-score hash:", OBSERVED_PRIMARY_HASH)


## LONG RUN — Primary conditional randomization

The next cell is the long-running primary Phase 7 Monte Carlo analysis.

It uses namespace 1, h = 20, Bmax = 5,058,108, batch size 20, exact replicate-ordered stopping, and an atomic checkpoint every 1,000 completed global replicates.

If execution is interrupted, rerun the notebook through this cell. The cell validates the checkpoint against the frozen family and observed-score hashes and resumes from the next uncommitted replicate.

Do not delete the checkpoint while the analysis is incomplete.


In [ ]:
# =============================================================================
# Initialize or restore primary Monte Carlo state
# =============================================================================

observed_rl_matrix = (
    primary_family[
        "C_RL"
    ]
    .to_numpy(dtype=np.float64)
    .reshape(
        len(PROGRAM_IDS),
        len(perturbagen_ids),
    )
    .T
)

if PRIMARY_CHECKPOINT_PATH.exists():
    restored = read_primary_checkpoint(
        PRIMARY_CHECKPOINT_PATH,
        family_hash=PRIMARY_FAMILY_HASH,
        observed_hash=OBSERVED_PRIMARY_HASH,
    )

    next_replicate_id = restored[
        "next_replicate_id"
    ]

    mc_g = restored[
        "mc_g"
    ]

    mc_L = restored[
        "mc_L"
    ]

    active_mask = restored[
        "active_mask"
    ]

    stop_code = restored[
        "stop_code"
    ]

    print(
        "PRIMARY MC: RESUMING FROM CHECKPOINT"
    )
    print(
        "Next replicate:",
        f"{next_replicate_id:,}",
    )
    print(
        "Active hypotheses:",
        f"{active_mask.sum():,}",
    )

else:
    next_replicate_id = 0

    mc_g = np.zeros(
        (
            len(perturbagen_ids),
            len(PROGRAM_IDS),
        ),
        dtype=np.int32,
    )

    mc_L = np.zeros_like(
        mc_g,
        dtype=np.int32,
    )

    active_mask = (
        active_primary_mask_2d.copy()
    )

    # stop_code:
    # 0 = not sampled / non-inverse / not-evaluable
    # 1 = stopped at h exceedances
    # 2 = reached Bmax with g < h
    stop_code = np.zeros(
        (
            len(perturbagen_ids),
            len(PROGRAM_IDS),
        ),
        dtype=np.int8,
    )

    print(
        "PRIMARY MC: STARTING NEW EXECUTION"
    )
    print(
        "Active hypotheses:",
        f"{active_mask.sum():,}",
    )


execution_start = time.perf_counter()



In [ ]:
# =============================================================================
# LONG RUN — Execute primary conditional randomization
# =============================================================================

while (
    next_replicate_id < FINAL_BMAX
    and active_mask.any()
):
    batch_start = int(
        next_replicate_id
    )

    batch_stop = min(
        batch_start + MC_BATCH_SIZE,
        FINAL_BMAX,
    )

    replicate_ids = np.arange(
        batch_start,
        batch_stop,
        dtype=np.int64,
    )

    active_perturbagen_union = (
        active_mask.any(axis=1)
    )

    active_perturbagen_ids = np.flatnonzero(
        active_perturbagen_union
    )

    active_signature_indices = np.concatenate(
        [
            signature_indices_by_perturbagen[
                int(pert_index)
            ]
            for pert_index in (
                active_perturbagen_ids
            )
        ]
    ).astype(
        np.int64,
        copy=False,
    )

    weight_tensor = (
        generate_pseudo_weight_batch(
            replicate_ids=replicate_ids,
            namespace=RNG_NAMESPACES[
                "primary_conditional_null"
            ],
        )
    )

    batch_size_actual = len(
        replicate_ids
    )

    active_signature_scores = np.empty(
        (
            len(active_signature_indices),
            batch_size_actual,
            len(PROGRAM_IDS),
        ),
        dtype=np.float64,
    )

    for local_start in range(
        0,
        len(active_signature_indices),
        SIGNATURE_BLOCK_SIZE,
    ):
        local_stop = min(
            local_start
            + SIGNATURE_BLOCK_SIZE,
            len(active_signature_indices),
        )

        signature_positions = (
            active_signature_indices[
                local_start:local_stop
            ]
        )

        signature_block = np.asarray(
            primary_bing_matrix[
                np.ix_(
                    query_bing_row_indices,
                    signature_positions,
                )
            ],
            dtype=np.float64,
        )

        active_signature_scores[
            local_start:local_stop,
            :,
            :,
        ] = score_signature_block_batch(
            signature_block=signature_block,
            signature_norms=(
                query_signature_norms[
                    signature_positions
                ]
            ),
            weight_tensor=weight_tensor,
        )

    flat_signature_scores = (
        active_signature_scores
        .reshape(
            len(active_signature_indices),
            batch_size_actual
            * len(PROGRAM_IDS),
        )
    )

    batch_perturbagen_raw = (
        aggregate_active_batch_to_perturbagen(
            active_signature_scores=(
                flat_signature_scores
            ),
            active_signature_indices=(
                active_signature_indices
            ),
            active_perturbagen_union=(
                active_perturbagen_union
            ),
        )
        .reshape(
            len(perturbagen_ids),
            batch_size_actual,
            len(PROGRAM_IDS),
        )
    )

    batch_perturbagen_rl = (
        batch_perturbagen_raw
        * resistance_like_multiplier_vector[
            np.newaxis,
            np.newaxis,
            :,
        ]
    )

    # Sequential bookkeeping remains exact and replicate ordered.
    for batch_offset, replicate_id in enumerate(
        replicate_ids
    ):
        active_before = (
            active_mask.copy()
        )

        if not active_before.any():
            break

        null_scores = (
            batch_perturbagen_rl[
                :,
                batch_offset,
                :,
            ]
        )

        exceedance = (
            null_scores
            <= observed_rl_matrix
        )

        mc_g[
            active_before
        ] += exceedance[
            active_before
        ].astype(np.int32)

        mc_L[
            active_before
        ] = int(
            replicate_id + 1
        )

        newly_stopped = (
            active_before
            & (
                mc_g
                >= MC_EXCEEDANCE_TARGET
            )
        )

        stop_code[
            newly_stopped
        ] = 1

        active_mask[
            newly_stopped
        ] = False

    next_replicate_id = int(
        batch_stop
    )

    if (
        next_replicate_id
        % CHECKPOINT_INTERVAL
        == 0
        or next_replicate_id
        == FINAL_BMAX
        or not active_mask.any()
    ):
        atomic_write_primary_checkpoint(
            path=PRIMARY_CHECKPOINT_PATH,
            next_replicate_id=(
                next_replicate_id
            ),
            mc_g=mc_g,
            mc_L=mc_L,
            active_mask=active_mask,
            stop_code=stop_code,
            family_hash=PRIMARY_FAMILY_HASH,
            observed_hash=OBSERVED_PRIMARY_HASH,
        )

        elapsed_hours = (
            time.perf_counter()
            - execution_start
        ) / 3600

        print(
            f"checkpoint replicate={next_replicate_id:,} "
            f"active={active_mask.sum():,} "
            f"elapsed_this_session={elapsed_hours:.2f} h"
        )




In [ ]:
# =============================================================================
# Finalize and validate primary Monte Carlo state
# =============================================================================

if (
    next_replicate_id >= FINAL_BMAX
    and active_mask.any()
):
    stop_code[
        active_mask
    ] = 2

    active_mask[
        :
    ] = False

    atomic_write_primary_checkpoint(
        path=PRIMARY_CHECKPOINT_PATH,
        next_replicate_id=FINAL_BMAX,
        mc_g=mc_g,
        mc_L=mc_L,
        active_mask=active_mask,
        stop_code=stop_code,
        family_hash=PRIMARY_FAMILY_HASH,
        observed_hash=OBSERVED_PRIMARY_HASH,
    )


if active_mask.any():
    raise RuntimeError(
        "Primary MC exited with active hypotheses."
    )

# Commit the terminal state even when every hypothesis stopped before Bmax
# or when no hypothesis entered the sequential branch.
atomic_write_primary_checkpoint(
    path=PRIMARY_CHECKPOINT_PATH,
    next_replicate_id=(
        next_replicate_id
    ),
    mc_g=mc_g,
    mc_L=mc_L,
    active_mask=active_mask,
    stop_code=stop_code,
    family_hash=PRIMARY_FAMILY_HASH,
    observed_hash=OBSERVED_PRIMARY_HASH,
)

sampled_mask = (
    active_primary_mask_2d
)

if not np.all(
    mc_L[
        sampled_mask
    ] > 0
):
    raise RuntimeError(
        "At least one inverse/evaluable hypothesis "
        "has no completed Monte Carlo draws."
    )

if np.any(
    mc_g[
        sampled_mask
    ] > MC_EXCEEDANCE_TARGET
):
    raise RuntimeError(
        "Sequential exceedance target was crossed."
    )

print("PRIMARY CONDITIONAL RANDOMIZATION: COMPLETE")
print("Maximum global replicate reached:", f"{next_replicate_id:,}")
print(
    "Stopped at h:",
    f"{(stop_code == 1).sum():,}",
)
print(
    "Reached Bmax with g<h:",
    f"{(stop_code == 2).sum():,}",
)
print(
    "Checkpoint:",
    project_relative_path(PRIMARY_CHECKPOINT_PATH),
)


In [ ]:
# =============================================================================
# Compute primary conditional randomization p-values
# =============================================================================

primary_perturbagen_connectivity = (
    primary_family.copy()
)

p_conditional_matrix = np.ones(
    (
        len(perturbagen_ids),
        len(PROGRAM_IDS),
    ),
    dtype=np.float64,
)

stopped_h = (
    stop_code == 1
)

reached_bmax = (
    stop_code == 2
)

p_conditional_matrix[
    stopped_h
] = (
    MC_EXCEEDANCE_TARGET
    / mc_L[
        stopped_h
    ].astype(np.float64)
)

p_conditional_matrix[
    reached_bmax
] = (
    mc_g[
        reached_bmax
    ].astype(np.float64)
    + 1.0
) / (
    FINAL_BMAX
    + 1.0
)

if np.any(
    p_conditional_matrix <= 0
):
    raise RuntimeError(
        "Zero Monte Carlo p-value encountered."
    )

primary_perturbagen_connectivity[
    "p_conditional"
] = (
    p_conditional_matrix.T.reshape(-1)
)

primary_perturbagen_connectivity[
    "p_for_multiplicity"
] = np.where(
    primary_perturbagen_connectivity[
        "directional_status"
    ].eq("INVERSE"),
    primary_perturbagen_connectivity[
        "p_conditional"
    ],
    1.0,
)

primary_perturbagen_connectivity[
    "mc_L"
] = mc_L.T.reshape(-1)

primary_perturbagen_connectivity[
    "mc_g"
] = mc_g.T.reshape(-1)

stop_reason_lookup = {
    0: "NOT_SAMPLED",
    1: "H_EXCEEDANCES",
    2: "BMAX_G_LT_H",
}

primary_perturbagen_connectivity[
    "mc_stop_reason"
] = [
    stop_reason_lookup[
        int(code)
    ]
    for code in (
        stop_code.T.reshape(-1)
    )
]

primary_perturbagen_connectivity[
    "Bmax"
] = FINAL_BMAX

primary_perturbagen_connectivity[
    "q_BY_global"
] = adjust_by(
    primary_perturbagen_connectivity[
        "p_for_multiplicity"
    ].to_numpy(dtype=np.float64)
)

primary_perturbagen_connectivity[
    "q_BH_global"
] = adjust_bh(
    primary_perturbagen_connectivity[
        "p_for_multiplicity"
    ].to_numpy(dtype=np.float64)
)




In [ ]:
# =============================================================================
# Apply global and descriptive multiplicity adjustments
# =============================================================================

# Program-specific multiplicity is descriptive only.
primary_perturbagen_connectivity[
    "q_BY_program_descriptive"
] = np.nan

primary_perturbagen_connectivity[
    "q_BH_program_descriptive"
] = np.nan

for program_id in PROGRAM_IDS:
    mask = (
        primary_perturbagen_connectivity[
            "program_id"
        ].eq(program_id)
    )

    program_p = (
        primary_perturbagen_connectivity.loc[
            mask,
            "p_for_multiplicity",
        ]
        .to_numpy(dtype=np.float64)
    )

    primary_perturbagen_connectivity.loc[
        mask,
        "q_BY_program_descriptive",
    ] = adjust_by(program_p)

    primary_perturbagen_connectivity.loc[
        mask,
        "q_BH_program_descriptive",
    ] = adjust_bh(program_p)




In [ ]:
# =============================================================================
# Assign frozen primary support states
# =============================================================================

primary_perturbagen_connectivity[
    "primary_support_status"
] = np.select(
    [
        primary_perturbagen_connectivity[
            "evaluability_status"
        ].eq("NOT_EVALUABLE"),
        primary_perturbagen_connectivity[
            "directional_status"
        ].eq("NON_INVERSE"),
        primary_perturbagen_connectivity[
            "q_BY_global"
        ].le(PRIMARY_BY_Q),
    ],
    [
        "NOT_EVALUABLE",
        "NON_INVERSE",
        "PRIMARY_BY_SUPPORTED",
    ],
    default="INVERSE_NOT_BY_SUPPORTED",
)




In [ ]:
# =============================================================================
# Persist primary lineage and perturbagen connectivity
# =============================================================================

PHASE7_OUTPUT_DIR = (
    Paths.perturbational_hypotheses
)

PHASE7_OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

LINEAGE_CONNECTIVITY_PATH = (
    PHASE7_OUTPUT_DIR
    / "701_lineage_connectivity.parquet"
)

PRIMARY_CONNECTIVITY_PATH = (
    PHASE7_OUTPUT_DIR
    / "701_primary_perturbagen_connectivity.csv"
)

lineage_connectivity.to_parquet(
    LINEAGE_CONNECTIVITY_PATH,
    index=False,
)

primary_perturbagen_connectivity.to_csv(
    PRIMARY_CONNECTIVITY_PATH,
    index=False,
)

print("PRIMARY INFERENCE: FINALIZED")
print(
    "PRIMARY_BY_SUPPORTED:",
    int(
        primary_perturbagen_connectivity[
            "primary_support_status"
        ].eq(
            "PRIMARY_BY_SUPPORTED"
        ).sum()
    ),
)
print(
    "INVERSE_NOT_BY_SUPPORTED:",
    int(
        primary_perturbagen_connectivity[
            "primary_support_status"
        ].eq(
            "INVERSE_NOT_BY_SUPPORTED"
        ).sum()
    ),
)
print(
    "NON_INVERSE:",
    int(
        primary_perturbagen_connectivity[
            "primary_support_status"
        ].eq(
            "NON_INVERSE"
        ).sum()
    ),
)
print(
    "NOT_EVALUABLE:",
    int(
        primary_perturbagen_connectivity[
            "primary_support_status"
        ].eq(
            "NOT_EVALUABLE"
        ).sum()
    ),
)
print(
    "Lineage output:",
    project_relative_path(
        LINEAGE_CONNECTIVITY_PATH
    ),
)
print(
    "Primary output:",
    project_relative_path(
        PRIMARY_CONNECTIVITY_PATH
    ),
)


## Prespecified sensitivity analyses

The following analyses are secondary to the frozen primary inference. They cannot create or rescue primary support.

They evaluate representation, technical-quality, lineage-composition, discrete-query, and competitive-null robustness while preserving the primary perturbagen-level status from the previous section.


In [ ]:
# =============================================================================
# Define subset hierarchy aggregation helper
# =============================================================================

def aggregate_subset_scores(
    metadata,
    score_matrix,
):
    score_matrix = np.asarray(
        score_matrix,
        dtype=np.float64,
    )

    score_columns = [
        f"score_{index}"
        for index in range(
            score_matrix.shape[1]
        )
    ]

    frame = (
        metadata[
            condition_keys
        ]
        .reset_index(drop=True)
        .copy()
    )

    for index, column in enumerate(
        score_columns
    ):
        frame[
            column
        ] = score_matrix[
            :,
            index
        ]

    condition = (
        frame
        .groupby(
            condition_keys,
            as_index=False,
            sort=True,
        )[score_columns]
        .median()
    )

    timepoint = (
        condition
        .groupby(
            timepoint_keys,
            as_index=False,
            sort=True,
        )[score_columns]
        .median()
    )

    cell_line = (
        timepoint
        .groupby(
            cell_line_keys,
            as_index=False,
            sort=True,
        )[score_columns]
        .median()
    )

    lineage = (
        cell_line
        .groupby(
            lineage_keys,
            as_index=False,
            sort=True,
        )[score_columns]
        .median()
    )

    perturbagen = (
        lineage
        .groupby(
            ["pert_id"],
            as_index=False,
            sort=True,
        )[score_columns]
        .median()
    )

    return {
        "condition": condition,
        "timepoint": timepoint,
        "cell_line": cell_line,
        "lineage": lineage,
        "perturbagen": perturbagen,
    }




In [ ]:
# =============================================================================
# Define sensitivity coverage helper
# =============================================================================

def subset_coverage(
    metadata,
    prefix,
):
    coverage = (
        metadata
        .groupby(
            "pert_id",
            as_index=False,
        )
        .agg(
            **{
                f"n_{prefix}_signatures": (
                    "sig_id",
                    "size",
                ),
                f"n_{prefix}_cell_lines": (
                    "cell_iname",
                    "nunique",
                ),
                f"n_{prefix}_lineages": (
                    "cell_lineage",
                    "nunique",
                ),
            }
        )
    )

    coverage[
        "sensitivity_evaluable"
    ] = (
        (
            coverage[
                f"n_{prefix}_cell_lines"
            ]
            >= PRIMARY_MIN_CELL_LINES
        )
        & (
            coverage[
                f"n_{prefix}_lineages"
            ]
            >= PRIMARY_MIN_LINEAGES
        )
    )

    return coverage




In [ ]:
# =============================================================================
# Define weighted subset scoring helper
# =============================================================================

def score_weighted_gene_subset(
    bing_rows,
    weight_matrix,
):
    bing_rows = np.asarray(
        bing_rows,
        dtype=np.int64,
    )

    weight_matrix = np.asarray(
        weight_matrix,
        dtype=np.float64,
    )

    weight_norms = np.linalg.norm(
        weight_matrix,
        axis=0,
    )

    signature_scores = np.full(
        (
            len(primary_signature_metadata),
            len(PROGRAM_IDS),
        ),
        np.nan,
        dtype=np.float64,
    )

    for start in range(
        0,
        len(primary_signature_metadata),
        SIGNATURE_BLOCK_SIZE,
    ):
        stop = min(
            start + SIGNATURE_BLOCK_SIZE,
            len(primary_signature_metadata),
        )

        block = np.asarray(
            primary_bing_matrix[
                bing_rows,
                start:stop,
            ],
            dtype=np.float64,
        )

        if not np.isfinite(
            block
        ).all():
            raise RuntimeError(
                "Sensitivity block contains non-finite values."
            )

        norms = np.linalg.norm(
            block,
            axis=0,
        )

        evaluable = norms > 0

        if evaluable.any():
            signature_scores[
                start:stop,
                :,
            ][
                evaluable,
                :
            ] = (
                (
                    weight_matrix.T
                    @ block[:, evaluable]
                )
                / (
                    weight_norms[
                        :,
                        np.newaxis,
                    ]
                    * norms[
                        evaluable
                    ][
                        np.newaxis,
                        :
                    ]
                )
            ).T

    return signature_scores




In [ ]:
# =============================================================================
# Prepare landmark-only weighted sensitivity
# =============================================================================

# -------------------------------------------------------------------------
# Landmark-only weighted representation.
# -------------------------------------------------------------------------

landmark_membership = (
    program_query_definitions.loc[
        program_query_definitions[
            "landmark_sensitivity_member"
        ].eq(True),
        [
            "program_id",
            "gene_id",
            "frozen_weight",
        ],
    ]
)

landmark_wide = (
    landmark_membership
    .pivot(
        index="gene_id",
        columns="program_id",
        values="frozen_weight",
    )
    .reindex(columns=list(PROGRAM_IDS))
    .sort_index()
)

if landmark_wide.shape != (
    90,
    len(PROGRAM_IDS),
):
    raise RuntimeError(
        "Unexpected landmark sensitivity query shape."
    )

if landmark_wide.isna().any().any():
    raise RuntimeError(
        "Landmark sensitivity support differs across programs."
    )

landmark_rows = (
    pd.Series(
        landmark_wide.index.to_numpy(
            dtype=np.int64
        )
    )
    .map(bing_row_lookup)
    .to_numpy(dtype=np.int64)
)

landmark_signature_raw = (
    score_weighted_gene_subset(
        landmark_rows,
        landmark_wide.to_numpy(
            dtype=np.float64
        ),
    )
)




In [ ]:
# =============================================================================
# Prepare stricter is_hiq sensitivity
# =============================================================================

# -------------------------------------------------------------------------
# is_hiq subset.
# -------------------------------------------------------------------------

hiq_mask = (
    primary_signature_metadata[
        "is_hiq"
    ].eq(1)
    .to_numpy()
)

hiq_metadata = (
    primary_signature_metadata.loc[
        hiq_mask
    ]
    .reset_index(drop=True)
)

hiq_raw_aggregation = (
    aggregate_subset_scores(
        hiq_metadata,
        observed_signature_raw[
            hiq_mask,
            :,
        ],
    )
)

hiq_coverage = subset_coverage(
    hiq_metadata,
    prefix="hiq",
)




In [ ]:
# =============================================================================
# Prepare pooled-cell-line composition sensitivity
# =============================================================================

# -------------------------------------------------------------------------
# Pooled-cell-line composition sensitivity.
# -------------------------------------------------------------------------

pooled_perturbagen_raw = (
    apply_median_reduction(
        observed_raw_aggregation[
            "cell_line"
        ],
        *build_median_reduction_plan(
            cell_line_to_perturbagen_direct
        ),
    )
)

pooled_perturbagen_rl = (
    pooled_perturbagen_raw
    * resistance_like_multiplier_vector[
        np.newaxis,
        :
    ]
)




In [ ]:
# =============================================================================
# Assemble landmark-only weighted sensitivity
# =============================================================================

sensitivity_rows = []

for program_index, program_id in enumerate(
    PROGRAM_IDS
):
    landmark_evaluable_mask = np.isfinite(
        landmark_signature_raw[
            :,
            program_index,
        ]
    )

    landmark_metadata_program = (
        primary_signature_metadata.loc[
            landmark_evaluable_mask
        ]
        .reset_index(drop=True)
    )

    landmark_program_agg = (
        aggregate_subset_scores(
            landmark_metadata_program,
            landmark_signature_raw[
                landmark_evaluable_mask,
                program_index,
            ][
                :,
                np.newaxis,
            ],
        )
    )

    landmark_coverage = subset_coverage(
        landmark_metadata_program,
        prefix="landmark",
    )

    landmark_program = (
        pd.DataFrame(
            {
                "pert_id": perturbagen_ids,
            }
        )
        .merge(
            landmark_program_agg[
                "perturbagen"
            ][
                [
                    "pert_id",
                    "score_0",
                ]
            ]
            .rename(
                columns={
                    "score_0": (
                        "sensitivity_score_raw"
                    )
                }
            ),
            on="pert_id",
            how="left",
            validate="one_to_one",
        )
        .merge(
            landmark_coverage,
            on="pert_id",
            how="left",
            validate="one_to_one",
        )
    )

    landmark_program[
        "sensitivity_evaluable"
    ] = (
        landmark_program[
            "sensitivity_evaluable"
        ]
        .fillna(False)
        .astype(bool)
    )

    landmark_program[
        "sensitivity_score_RL"
    ] = (
        landmark_program[
            "sensitivity_score_raw"
        ]
        * resistance_like_multiplier_vector[
            program_index
        ]
    )

    landmark_program.loc[
        ~landmark_program[
            "sensitivity_evaluable"
        ],
        [
            "sensitivity_score_raw",
            "sensitivity_score_RL",
        ],
    ] = np.nan

    landmark_program[
        "sensitivity_type"
    ] = "landmark_weighted_cosine"

    landmark_program[
        "program_id"
    ] = program_id

    sensitivity_rows.append(
        landmark_program
    )


In [ ]:
# =============================================================================
# Assemble pooled-cell-line composition sensitivity
# =============================================================================

for program_index, program_id in enumerate(
    PROGRAM_IDS
):
    sensitivity_rows.append(
        pd.DataFrame(
            {
                "sensitivity_type": (
                    "pooled_cell_line_weighted_cosine"
                ),
                "program_id": program_id,
                "pert_id": perturbagen_ids,
                "sensitivity_score_raw": (
                    pooled_perturbagen_raw[
                        :,
                        program_index,
                    ]
                ),
                "sensitivity_score_RL": (
                    pooled_perturbagen_rl[
                        :,
                        program_index,
                    ]
                ),
                "sensitivity_evaluable": True,
            }
        )
    )


In [ ]:
# =============================================================================
# Assemble stricter is_hiq sensitivity
# =============================================================================

for program_index, program_id in enumerate(
    PROGRAM_IDS
):
    hiq_program = (
        pd.DataFrame(
            {
                "pert_id": perturbagen_ids,
            }
        )
        .merge(
            hiq_raw_aggregation[
                "perturbagen"
            ][
                [
                    "pert_id",
                    f"score_{program_index}",
                ]
            ]
            .rename(
                columns={
                    f"score_{program_index}": (
                        "sensitivity_score_raw"
                    )
                }
            ),
            on="pert_id",
            how="left",
            validate="one_to_one",
        )
        .merge(
            hiq_coverage,
            on="pert_id",
            how="left",
            validate="one_to_one",
        )
    )

    hiq_program[
        "sensitivity_evaluable"
    ] = (
        hiq_program[
            "sensitivity_evaluable"
        ]
        .fillna(False)
        .astype(bool)
    )

    hiq_program[
        "sensitivity_score_RL"
    ] = (
        hiq_program[
            "sensitivity_score_raw"
        ]
        * resistance_like_multiplier_vector[
            program_index
        ]
    )

    hiq_program.loc[
        ~hiq_program[
            "sensitivity_evaluable"
        ],
        [
            "sensitivity_score_raw",
            "sensitivity_score_RL",
        ],
    ] = np.nan

    hiq_program[
        "sensitivity_type"
    ] = "is_hiq_weighted_cosine"

    hiq_program[
        "program_id"
    ] = program_id

    sensitivity_rows.append(
        hiq_program
    )

sensitivity_connectivity = pd.concat(
    sensitivity_rows,
    ignore_index=True,
    sort=False,
)

print("WEIGHTED / QUALITY / COMPOSITION SENSITIVITIES: PASS")
print(
    sensitivity_connectivity[
        "sensitivity_type"
    ].value_counts()
)


In [ ]:
# =============================================================================
# Freeze CMap-style sensitivity dimensions
# =============================================================================

CMAP_RANK_BLOCK_SIZE = 64
N_BING = len(bing_gene_table)

if not bing_gene_table[
    "gene_symbol"
].dropna().is_unique:
    raise RuntimeError(
        "BING gene symbols are not unique; WTCS mapping is ambiguous."
    )




In [ ]:
# =============================================================================
# Define CMap-style enrichment-score helper
# =============================================================================

def enrichment_score_from_hits(
    hit_ranks,
    hit_abs_values,
    n_genes,
):
    hit_ranks = np.asarray(
        hit_ranks,
        dtype=np.int64,
    )

    hit_abs_values = np.asarray(
        hit_abs_values,
        dtype=np.float64,
    )

    sort_index = np.argsort(
        hit_ranks,
        axis=1,
        kind="stable",
    )

    ranks = np.take_along_axis(
        hit_ranks,
        sort_index,
        axis=1,
    )

    weights = np.take_along_axis(
        hit_abs_values,
        sort_index,
        axis=1,
    )

    denominators = weights.sum(
        axis=1
    )

    evaluable = denominators > 0

    es = np.full(
        len(ranks),
        np.nan,
        dtype=np.float64,
    )

    if not evaluable.any():
        return es, evaluable

    ranks_e = ranks[
        evaluable,
        :
    ]

    weights_e = weights[
        evaluable,
        :
    ] / denominators[
        evaluable,
        np.newaxis,
    ]

    n_hits = ranks.shape[1]

    miss_step = (
        1.0
        / (
            n_genes
            - n_hits
        )
    )

    cumulative_hits = np.cumsum(
        weights_e,
        axis=1,
    )

    hit_number = np.arange(
        n_hits,
        dtype=np.int64,
    )[
        np.newaxis,
        :
    ]

    misses_before = (
        ranks_e
        - hit_number
    )

    after_values = (
        cumulative_hits
        - misses_before
        * miss_step
    )

    previous_cumulative = np.concatenate(
        [
            np.zeros(
                (
                    len(ranks_e),
                    1,
                ),
                dtype=np.float64,
            ),
            cumulative_hits[
                :,
                :-1,
            ],
        ],
        axis=1,
    )

    before_values = (
        previous_cumulative
        - misses_before
        * miss_step
    )

    before_positions = (
        ranks_e - 1
    )

    before_valid = (
        before_positions >= 0
    )

    candidate_values = np.concatenate(
        [
            before_values,
            after_values,
        ],
        axis=1,
    )

    candidate_positions = np.concatenate(
        [
            before_positions,
            ranks_e,
        ],
        axis=1,
    )

    candidate_valid = np.concatenate(
        [
            before_valid,
            np.ones_like(
                after_values,
                dtype=bool,
            ),
        ],
        axis=1,
    )

    candidate_abs = np.where(
        candidate_valid,
        np.abs(candidate_values),
        -np.inf,
    )

    max_abs = candidate_abs.max(
        axis=1
    )

    is_max = (
        candidate_abs
        == max_abs[
            :,
            np.newaxis,
        ]
    )

    earliest_position = np.where(
        is_max,
        candidate_positions,
        n_genes + 1,
    ).min(axis=1)

    selected = (
        is_max
        & (
            candidate_positions
            == earliest_position[
                :,
                np.newaxis,
            ]
        )
    )

    selected_column = np.argmax(
        selected,
        axis=1,
    )

    es[
        evaluable
    ] = candidate_values[
        np.arange(
            len(candidate_values)
        ),
        selected_column,
    ]

    return es, evaluable




In [ ]:
# =============================================================================
# Validate enrichment-score helper
# =============================================================================

# Tiny independent ES sanity checks.
toy_hit_ranks = np.array(
    [
        [0, 2],
        [2, 3],
    ],
    dtype=np.int64,
)

toy_hit_weights = np.ones(
    (2, 2),
    dtype=np.float64,
)

toy_es, toy_eval = (
    enrichment_score_from_hits(
        toy_hit_ranks,
        toy_hit_weights,
        n_genes=4,
    )
)

if not toy_eval.all():
    raise RuntimeError(
        "WTCS ES toy validation unexpectedly non-evaluable."
    )

np.testing.assert_allclose(
    toy_es,
    np.array(
        [0.5, -1.0],
        dtype=np.float64,
    ),
    rtol=NUMERICAL_RTOL,
    atol=NUMERICAL_ATOL,
)




In [ ]:
# =============================================================================
# Map fixed CMap-style 150/150 query arms
# =============================================================================

cmap_arm_rows = {}

for program_id in PROGRAM_IDS:
    program_rows = (
        program_query_definitions.loc[
            program_query_definitions[
                "program_id"
            ].eq(program_id)
        ]
    )

    cmap_arm_rows[
        program_id
    ] = {}

    for arm in (
        "up",
        "down",
    ):
        genes = (
            program_rows.loc[
                program_rows[
                    "cmap_style_arm"
                ].eq(arm),
                "gene_id",
            ]
            .astype(np.int64)
            .to_numpy()
        )

        if len(genes) != 150:
            raise RuntimeError(
                f"{program_id} {arm} arm is not 150 genes."
            )

        rows = (
            pd.Series(genes)
            .map(bing_row_lookup)
        )

        if rows.isna().any():
            raise RuntimeError(
                "CMap-style query gene absent from BING cache."
            )

        cmap_arm_rows[
            program_id
        ][arm] = rows.to_numpy(
            dtype=np.int64
        )




In [ ]:
# =============================================================================
# Compute signature-level raw WTCS
# =============================================================================

cmap_signature_raw = np.full(
    (
        len(primary_signature_metadata),
        len(PROGRAM_IDS),
    ),
    np.nan,
    dtype=np.float64,
)

for start in range(
    0,
    len(primary_signature_metadata),
    CMAP_RANK_BLOCK_SIZE,
):
    stop = min(
        start + CMAP_RANK_BLOCK_SIZE,
        len(primary_signature_metadata),
    )

    block = np.asarray(
        primary_bing_matrix[
            :,
            start:stop,
        ],
        dtype=np.float64,
    )

    order = np.argsort(
        -block,
        axis=0,
        kind="stable",
    )

    ranks = np.empty_like(
        order,
        dtype=np.int64,
    )

    np.put_along_axis(
        ranks,
        order,
        np.arange(
            N_BING,
            dtype=np.int64,
        )[
            :,
            np.newaxis,
        ],
        axis=0,
    )

    for program_index, program_id in enumerate(
        PROGRAM_IDS
    ):
        arm_es = {}

        for arm in (
            "up",
            "down",
        ):
            arm_rows = (
                cmap_arm_rows[
                    program_id
                ][arm]
            )

            hit_ranks = (
                ranks[
                    arm_rows,
                    :,
                ].T
            )

            hit_abs_values = np.abs(
                block[
                    arm_rows,
                    :,
                ].T
            )

            arm_es[
                arm
            ], _ = (
                enrichment_score_from_hits(
                    hit_ranks,
                    hit_abs_values,
                    n_genes=N_BING,
                )
            )

        up = arm_es["up"]
        down = arm_es["down"]

        evaluable = (
            np.isfinite(up)
            & np.isfinite(down)
        )

        wtcs = np.full(
            stop - start,
            np.nan,
            dtype=np.float64,
        )

        opposite = (
            evaluable
            & (
                up * down < 0
            )
        )

        same_direction = (
            evaluable
            & ~opposite
        )

        wtcs[
            opposite
        ] = (
            up[
                opposite
            ]
            - down[
                opposite
            ]
        ) / 2.0

        wtcs[
            same_direction
        ] = 0.0

        cmap_signature_raw[
            start:stop,
            program_index,
        ] = wtcs

    if (
        start == 0
        or stop
        == len(primary_signature_metadata)
        or (
            stop
            % 25000
            < CMAP_RANK_BLOCK_SIZE
        )
    ):
        print(
            "WTCS signatures:",
            f"{stop:,}/"
            f"{len(primary_signature_metadata):,}",
        )




In [ ]:
# =============================================================================
# Aggregate CMap-style sensitivity with explicit evaluability
# =============================================================================

for program_index, program_id in enumerate(
    PROGRAM_IDS
):
    cmap_evaluable_mask = np.isfinite(
        cmap_signature_raw[
            :,
            program_index,
        ]
    )

    cmap_metadata_program = (
        primary_signature_metadata.loc[
            cmap_evaluable_mask
        ]
        .reset_index(drop=True)
    )

    cmap_program_agg = (
        aggregate_subset_scores(
            cmap_metadata_program,
            cmap_signature_raw[
                cmap_evaluable_mask,
                program_index,
            ][
                :,
                np.newaxis,
            ],
        )
    )

    cmap_coverage = subset_coverage(
        cmap_metadata_program,
        prefix="cmap",
    )

    cmap_program = (
        pd.DataFrame(
            {
                "pert_id": perturbagen_ids,
            }
        )
        .merge(
            cmap_program_agg[
                "perturbagen"
            ][
                [
                    "pert_id",
                    "score_0",
                ]
            ]
            .rename(
                columns={
                    "score_0": (
                        "sensitivity_score_raw"
                    )
                }
            ),
            on="pert_id",
            how="left",
            validate="one_to_one",
        )
        .merge(
            cmap_coverage,
            on="pert_id",
            how="left",
            validate="one_to_one",
        )
    )

    cmap_program[
        "sensitivity_evaluable"
    ] = (
        cmap_program[
            "sensitivity_evaluable"
        ]
        .fillna(False)
        .astype(bool)
    )

    cmap_program[
        "sensitivity_score_RL"
    ] = (
        cmap_program[
            "sensitivity_score_raw"
        ]
        * resistance_like_multiplier_vector[
            program_index
        ]
    )

    cmap_program.loc[
        ~cmap_program[
            "sensitivity_evaluable"
        ],
        [
            "sensitivity_score_raw",
            "sensitivity_score_RL",
        ],
    ] = np.nan

    cmap_program[
        "sensitivity_type"
    ] = "cmap_style_raw_wtcs"

    cmap_program[
        "program_id"
    ] = program_id

    sensitivity_connectivity = pd.concat(
        [
            sensitivity_connectivity,
            cmap_program,
        ],
        ignore_index=True,
        sort=False,
    )


print("CMAP-STYLE RAW WTCS SENSITIVITY: PASS")
print(
    "Signature-level non-evaluable counts:",
    {
        program_id: int(
            np.isnan(
                cmap_signature_raw[
                    :,
                    program_index,
                ]
            ).sum()
        )
        for program_index, program_id in enumerate(
            PROGRAM_IDS
        )
    },
)


In [ ]:
# =============================================================================
# Prepare competitive matched pseudo-program sensitivity
# =============================================================================

COMPETITIVE_DRAWS = 10000
COMPETITIVE_CHECKPOINT_INTERVAL = 1000
COMPETITIVE_CHECKPOINT_PATH = (
    Paths.perturbational_hypotheses
    / "701_competitive_null_checkpoint.npz"
)

supported_primary = (
    primary_perturbagen_connectivity.loc[
        primary_perturbagen_connectivity[
            "primary_support_status"
        ].eq("PRIMARY_BY_SUPPORTED")
    ]
    .copy()
)

competitive_rows = []
competitive_null_required = (
    not supported_primary.empty
)

if not competitive_null_required:
    print(
        "COMPETITIVE NULL: SKIPPED — "
        "no PRIMARY_BY_SUPPORTED hypotheses"
    )
else:
    supported_mask = np.zeros(
        (
            len(perturbagen_ids),
            len(PROGRAM_IDS),
        ),
        dtype=bool,
    )

    for row in supported_primary.itertuples(
        index=False
    ):
        supported_mask[
            int(row.perturbagen_index),
            int(row.program_index),
        ] = True

    supported_pert_union = (
        supported_mask.any(axis=1)
    )

    supported_signature_indices = (
        np.concatenate(
            [
                signature_indices_by_perturbagen[
                    int(pert_index)
                ]
                for pert_index in np.flatnonzero(
                    supported_pert_union
                )
            ]
        )
        .astype(
            np.int64,
            copy=False,
        )
    )

    competitive_g = np.zeros(
        (
            len(perturbagen_ids),
            len(PROGRAM_IDS),
        ),
        dtype=np.int32,
    )

    competitive_next_draw = 0

    supported_key_text = "\n".join(
        supported_primary[
            ["program_id", "pert_id"]
        ]
        .astype(str)
        .agg("\t".join, axis=1)
        .tolist()
    )

    supported_hash = hashlib.sha256(
        supported_key_text.encode(
            "utf-8"
        )
    ).hexdigest()

    if COMPETITIVE_CHECKPOINT_PATH.exists():
        with np.load(
            COMPETITIVE_CHECKPOINT_PATH,
            allow_pickle=False,
        ) as checkpoint:
            if str(
                checkpoint[
                    "supported_hash"
                ][0]
            ) != supported_hash:
                raise RuntimeError(
                    "Competitive checkpoint support-set mismatch."
                )

            if str(
                checkpoint[
                    "competitive_plan_hash"
                ][0]
            ) != COMPETITIVE_PLAN_HASH:
                raise RuntimeError(
                    "Competitive checkpoint matching-plan mismatch."
                )

            competitive_next_draw = int(
                checkpoint[
                    "next_draw"
                ][0]
            )

            competitive_g = checkpoint[
                "g"
            ].copy()

        print(
            "COMPETITIVE NULL: RESUMING",
            competitive_next_draw,
        )

print(
    "Competitive-null hypotheses:",
    f"{len(supported_primary):,}",
)


In [ ]:
# =============================================================================
# Define one competitive pseudo-program draw
# =============================================================================

def score_competitive_pseudoprogram_draw(
    draw_id,
):
    selected_bing_rows = np.empty(
        len(query_gene_ids),
        dtype=np.int64,
    )

    for block in competitive_blocks:
        query_rows = np.asarray(
            block[
                "query_row_indices"
            ],
            dtype=np.int64,
        )

        background_gene_ids = np.asarray(
            block[
                "eligible_background_gene_ids"
            ],
            dtype=np.int64,
        )

        rng = make_phase7_rng(
            namespace=RNG_NAMESPACES[
                "competitive_null_sensitivity"
            ],
            replicate_id=draw_id,
            stratum_id=int(
                block[
                    "rng_stratum_id"
                ]
            ),
        )

        selected_gene_ids = (
            background_gene_ids[
                rng.permutation(
                    len(background_gene_ids)
                )[
                    : len(query_rows)
                ]
            ]
        )

        selected_rows = (
            pd.Series(
                selected_gene_ids
            )
            .map(
                bing_row_lookup
            )
        )

        if selected_rows.isna().any():
            raise RuntimeError(
                "Competitive background gene missing from BING cache."
            )

        selected_bing_rows[
            query_rows
        ] = selected_rows.to_numpy(
            dtype=np.int64
        )

    signature_scores = np.empty(
        (
            len(
                supported_signature_indices
            ),
            len(PROGRAM_IDS),
        ),
        dtype=np.float64,
    )

    for local_start in range(
        0,
        len(
            supported_signature_indices
        ),
        SIGNATURE_BLOCK_SIZE,
    ):
        local_stop = min(
            local_start
            + SIGNATURE_BLOCK_SIZE,
            len(
                supported_signature_indices
            ),
        )

        signature_positions = (
            supported_signature_indices[
                local_start:local_stop
            ]
        )

        block = np.asarray(
            primary_bing_matrix[
                np.ix_(
                    selected_bing_rows,
                    signature_positions,
                )
            ],
            dtype=np.float64,
        )

        norms = np.linalg.norm(
            block,
            axis=0,
        )

        if not np.all(
            norms > 0
        ):
            raise RuntimeError(
                "Competitive pseudo-program encountered zero norm."
            )

        signature_scores[
            local_start:local_stop,
            :,
        ] = (
            (
                frozen_weight_matrix.T
                @ block
            )
            / (
                frozen_weight_norms[
                    :,
                    np.newaxis,
                ]
                * norms[
                    np.newaxis,
                    :
                ]
            )
        ).T

    competitive_pert_raw = (
        aggregate_active_batch_to_perturbagen(
            active_signature_scores=(
                signature_scores
            ),
            active_signature_indices=(
                supported_signature_indices
            ),
            active_perturbagen_union=(
                supported_pert_union
            ),
        )
    )

    return (
        competitive_pert_raw
        * resistance_like_multiplier_vector[
            np.newaxis,
            :
        ]
    )


In [ ]:
# =============================================================================
# Execute 10,000 competitive pseudo-program draws
# =============================================================================

if competitive_null_required:
    for draw_id in range(
        competitive_next_draw,
        COMPETITIVE_DRAWS,
    ):
        competitive_pert_rl = (
            score_competitive_pseudoprogram_draw(
                draw_id
            )
        )

        competitive_exceedance = (
            competitive_pert_rl
            <= observed_rl_matrix
        )

        competitive_g[
            supported_mask
        ] += competitive_exceedance[
            supported_mask
        ].astype(np.int32)

        completed = draw_id + 1

        if (
            completed
            % COMPETITIVE_CHECKPOINT_INTERVAL
            == 0
            or completed
            == COMPETITIVE_DRAWS
        ):
            temp_path = (
                COMPETITIVE_CHECKPOINT_PATH
                .with_suffix(
                    ".npz.tmp"
                )
            )

            with temp_path.open(
                "wb"
            ) as handle:
                np.savez(
                    handle,
                    supported_hash=np.array(
                        [supported_hash]
                    ),
                    competitive_plan_hash=np.array(
                        [COMPETITIVE_PLAN_HASH]
                    ),
                    next_draw=np.array(
                        [completed],
                        dtype=np.int64,
                    ),
                    g=competitive_g,
                )

                handle.flush()
                os.fsync(
                    handle.fileno()
                )

            os.replace(
                temp_path,
                COMPETITIVE_CHECKPOINT_PATH,
            )

            print(
                "competitive checkpoint:",
                f"{completed:,}/"
                f"{COMPETITIVE_DRAWS:,}",
            )


In [ ]:
# =============================================================================
# Finalize competitive-null sensitivity output
# =============================================================================

if competitive_null_required:
    for row in supported_primary.itertuples(
        index=False
    ):
        g = int(
            competitive_g[
                int(row.perturbagen_index),
                int(row.program_index),
            ]
        )

        competitive_rows.append(
            {
                "sensitivity_type": (
                    "competitive_matched_pseudoprogram"
                ),
                "program_id": row.program_id,
                "pert_id": row.pert_id,
                "sensitivity_score_raw": np.nan,
                "sensitivity_score_RL": np.nan,
                "sensitivity_evaluable": True,
                "competitive_draws": (
                    COMPETITIVE_DRAWS
                ),
                "competitive_g": g,
                "competitive_null_tail_fraction": (
                    (g + 1.0)
                    / (
                        COMPETITIVE_DRAWS
                        + 1.0
                    )
                ),
            }
        )

    sensitivity_connectivity = pd.concat(
        [
            sensitivity_connectivity,
            pd.DataFrame(
                competitive_rows
            ),
        ],
        ignore_index=True,
        sort=False,
    )

    print(
        "COMPETITIVE NULL SENSITIVITY: COMPLETE"
    )
    print(
        "Supported hypotheses evaluated:",
        f"{len(supported_primary):,}",
    )


## Perturbational-context diagnostics

The remaining diagnostics are contextual only. Provider activity/reproducibility fields and the five frozen Hallmark scores do not modify primary eligibility, p-values, q-values, or support status.


In [ ]:
# =============================================================================
# Define provider diagnostic aggregation helper
# =============================================================================

PROVIDER_DIAGNOSTICS = (
    "tas",
    "ss_ngene",
    "cc_q75",
)


def aggregate_diagnostic_metric(
    metric_name,
):
    values = pd.to_numeric(
        primary_signature_metadata[
            metric_name
        ],
        errors="coerce",
    ).to_numpy(dtype=np.float64)

    finite = np.isfinite(values)

    diagnostic_metadata = (
        primary_signature_metadata.loc[
            finite
        ]
        .reset_index(drop=True)
    )

    diagnostic_scores = (
        values[
            finite
        ][
            :,
            np.newaxis,
        ]
    )

    aggregated = aggregate_subset_scores(
        diagnostic_metadata,
        diagnostic_scores,
    )

    final = (
        aggregated[
            "perturbagen"
        ][
            [
                "pert_id",
                "score_0",
            ]
        ]
        .rename(
            columns={
                "score_0": (
                    "diagnostic_value"
                )
            }
        )
    )

    counts = (
        diagnostic_metadata
        .groupby(
            "pert_id",
            as_index=False,
        )
        .agg(
            n_evaluable_signatures=(
                "sig_id",
                "size",
            ),
            n_evaluable_cell_lines=(
                "cell_iname",
                "nunique",
            ),
            n_evaluable_lineages=(
                "cell_lineage",
                "nunique",
            ),
        )
    )

    return (
        pd.DataFrame(
            {
                "pert_id": perturbagen_ids,
            }
        )
        .merge(
            final,
            on="pert_id",
            how="left",
            validate="one_to_one",
        )
        .merge(
            counts,
            on="pert_id",
            how="left",
            validate="one_to_one",
        )
        .assign(
            diagnostic_type=(
                "provider_activity_reproducibility"
            ),
            diagnostic_name=metric_name,
        )
    )




In [ ]:
# =============================================================================
# Aggregate provider activity and reproducibility diagnostics
# =============================================================================

diagnostic_frames = [
    aggregate_diagnostic_metric(
        metric_name
    )
    for metric_name in (
        PROVIDER_DIAGNOSTICS
    )
]

print(
    "PROVIDER DIAGNOSTICS: AGGREGATED",
    PROVIDER_DIAGNOSTICS,
)


In [ ]:
# =============================================================================
# Load and validate frozen Hallmark diagnostic panel
# =============================================================================

FROZEN_HALLMARKS = (
    "HALLMARK_E2F_TARGETS",
    "HALLMARK_G2M_CHECKPOINT",
    "HALLMARK_P53_PATHWAY",
    "HALLMARK_APOPTOSIS",
    "HALLMARK_UNFOLDED_PROTEIN_RESPONSE",
)

hallmark_sets = {}

with HALLMARK_GMT_PATH.open(
    "r",
    encoding="utf-8",
) as handle:
    for line in handle:
        fields = line.rstrip(
            "\n"
        ).split("\t")

        if len(fields) < 3:
            continue

        hallmark_name = fields[0]

        if hallmark_name in (
            FROZEN_HALLMARKS
        ):
            hallmark_sets[
                hallmark_name
            ] = set(
                fields[2:]
            )

if set(hallmark_sets) != set(
    FROZEN_HALLMARKS
):
    raise RuntimeError(
        "Frozen Hallmark panel not fully present in GMT."
    )

bing_symbol_lookup = (
    bing_gene_table
    .reset_index(
        names="bing_row"
    )
    .set_index(
        "gene_symbol"
    )[
        "bing_row"
    ]
)

program_support_symbols = {
    program_id: set(
        program_query_definitions.loc[
            program_query_definitions[
                "program_id"
            ].eq(program_id),
            "gene_symbol",
        ].dropna()
    )
    for program_id in PROGRAM_IDS
}



In [ ]:
# =============================================================================
# Score and aggregate frozen Hallmark diagnostics
# =============================================================================

hallmark_metadata_rows = []

for hallmark_name in FROZEN_HALLMARKS:
    total_symbols = hallmark_sets[
        hallmark_name
    ]

    represented_symbols = sorted(
        total_symbols
        & set(
            bing_symbol_lookup.index
        )
    )

    bing_rows = (
        bing_symbol_lookup.loc[
            represented_symbols
        ]
        .to_numpy(dtype=np.int64)
    )

    if len(bing_rows) == 0:
        raise RuntimeError(
            f"No BING genes for {hallmark_name}."
        )

    hallmark_signature_scores = np.empty(
        len(primary_signature_metadata),
        dtype=np.float64,
    )

    for start in range(
        0,
        len(primary_signature_metadata),
        SIGNATURE_BLOCK_SIZE,
    ):
        stop = min(
            start + SIGNATURE_BLOCK_SIZE,
            len(primary_signature_metadata),
        )

        block = np.asarray(
            primary_bing_matrix[
                bing_rows,
                start:stop,
            ],
            dtype=np.float64,
        )

        if not np.isfinite(
            block
        ).all():
            raise RuntimeError(
                f"Non-finite Hallmark values: {hallmark_name}"
            )

        hallmark_signature_scores[
            start:stop
        ] = block.mean(
            axis=0,
            dtype=np.float64,
        )

    hallmark_agg = (
        aggregate_signature_scores(
            hallmark_signature_scores
        )
    )

    diagnostic_frames.append(
        pd.DataFrame(
            {
                "pert_id": perturbagen_ids,
                "diagnostic_value": (
                    hallmark_agg[
                        "perturbagen"
                    ]
                ),
                "n_evaluable_signatures": (
                    primary_perturbagen_coverage
                    .set_index("pert_id")
                    .loc[
                        perturbagen_ids,
                        "n_signatures",
                    ]
                    .to_numpy()
                ),
                "n_evaluable_cell_lines": (
                    primary_perturbagen_coverage
                    .set_index("pert_id")
                    .loc[
                        perturbagen_ids,
                        "n_cell_lines",
                    ]
                    .to_numpy()
                ),
                "n_evaluable_lineages": (
                    primary_perturbagen_coverage
                    .set_index("pert_id")
                    .loc[
                        perturbagen_ids,
                        "n_lineages",
                    ]
                    .to_numpy()
                ),
                "diagnostic_type": (
                    "hallmark_mean_signed_z"
                ),
                "diagnostic_name": (
                    hallmark_name
                ),
            }
        )
    )

    metadata_row = {
        "hallmark": hallmark_name,
        "total_hallmark_genes": int(
            len(total_symbols)
        ),
        "bing_represented_genes": int(
            len(represented_symbols)
        ),
        "bing_coverage_fraction": float(
            len(represented_symbols)
            / len(total_symbols)
        ),
    }

    for program_id in PROGRAM_IDS:
        metadata_row[
            f"overlap_{program_id}"
        ] = int(
            len(
                total_symbols
                & program_support_symbols[
                    program_id
                ]
            )
        )

    hallmark_metadata_rows.append(
        metadata_row
    )


hallmark_diagnostic_metadata = pd.DataFrame(
    hallmark_metadata_rows
)

perturbational_context_diagnostics = (
    pd.concat(
        diagnostic_frames,
        ignore_index=True,
        sort=False,
    )
    .sort_values(
        [
            "diagnostic_type",
            "diagnostic_name",
            "pert_id",
        ],
        kind="mergesort",
    )
    .reset_index(drop=True)
)

print("HALLMARK DIAGNOSTICS: PASS")
display(hallmark_diagnostic_metadata)
print(
    "Diagnostic rows:",
    f"{len(perturbational_context_diagnostics):,}",
)


## Stable handoff, provenance, and closure

The remaining cells persist the five frozen Notebook 701 outputs, independently re-read and fingerprint them, register their provenance, and run final closure QA. No downstream prioritization occurs in Notebook 701.


In [ ]:
# =============================================================================
# Persist sensitivity and diagnostic handoffs
# =============================================================================

SENSITIVITY_CONNECTIVITY_PATH = (
    PHASE7_OUTPUT_DIR
    / "701_sensitivity_connectivity.parquet"
)

CONTEXT_DIAGNOSTICS_PATH = (
    PHASE7_OUTPUT_DIR
    / "701_perturbational_context_diagnostics.parquet"
)

ANALYSIS_METADATA_PATH = (
    PHASE7_OUTPUT_DIR
    / "701_analysis_metadata.json"
)

sensitivity_connectivity = (
    sensitivity_connectivity
    .sort_values(
        [
            "sensitivity_type",
            "program_id",
            "pert_id",
        ],
        kind="mergesort",
        na_position="last",
    )
    .reset_index(drop=True)
)

sensitivity_connectivity.to_parquet(
    SENSITIVITY_CONNECTIVITY_PATH,
    index=False,
)

perturbational_context_diagnostics.to_parquet(
    CONTEXT_DIAGNOSTICS_PATH,
    index=False,
)



In [ ]:
# =============================================================================
# Prepare Notebook 701 provenance context
# =============================================================================

contract_path = (
    Paths.root
    / "docs"
    / "contracts"
    / "phase7"
    / "PHASE7_ANALYSIS_CONTRACT.md"
)

software_versions = {
    "python": platform.python_version(),
    "numpy": np.__version__,
    "pandas": pd.__version__,
    "h5py": h5py.__version__,
    "threadpoolctl": package_version(
        "threadpoolctl"
    ),
}

try:
    software_versions[
        "pyarrow"
    ] = package_version(
        "pyarrow"
    )
except Exception:
    software_versions[
        "pyarrow"
    ] = None

primary_status_counts = (
    primary_perturbagen_connectivity[
        "primary_support_status"
    ]
    .value_counts()
    .to_dict()
)



In [ ]:
# =============================================================================
# Build Notebook 701 analysis metadata
# =============================================================================

analysis_metadata = {
    "schema_version": 1,
    "notebook": "701_connectivity_analysis",
    "analysis_role": (
        "phase7_primary_perturbational_connectivity_"
        "conditional_randomization_and_sensitivity_analysis"
    ),
    "execution_date": "2026-09-29",
    "contract": {
        "path": project_relative_path(
            contract_path
        ),
        "sha256": calculate_sha256(
            contract_path
        ),
        "prospective_bmax_amendment": (
            "2026-09-29"
        ),
    },
    "input_artifact_hashes": input_hashes,
    "program_order": list(
        PROGRAM_IDS
    ),
    "primary_universe": {
        "signatures": int(
            len(primary_signature_metadata)
        ),
        "perturbagens": int(
            len(perturbagen_ids)
        ),
        "hypotheses": int(
            len(
                primary_perturbagen_connectivity
            )
        ),
        "minimum_cell_lines": (
            PRIMARY_MIN_CELL_LINES
        ),
        "minimum_lineages": (
            PRIMARY_MIN_LINEAGES
        ),
    },
    "connectivity": {
        "primary_gene_space": "BING",
        "query_genes_per_program": 1495,
        "score": "signed weighted cosine",
        "resistance_like_multipliers": {
            program_id: int(multiplier)
            for program_id, multiplier in zip(
                PROGRAM_IDS,
                resistance_like_multiplier_vector,
            )
        },
        "hierarchy": [
            "signature",
            "condition",
            "timepoint",
            "cell_line",
            "lineage",
            "perturbagen",
        ],
        "cross_cell_aggregation": (
            "equal-lineage median"
        ),
        "lineage_dispersion": (
            "IQR of lineage-level resistance-like-oriented "
            "connectivity"
        ),
    },
    "conditional_randomization": {
        "interpretation": (
            "conditional assignment-specificity null"
        ),
        "exchangeability_assumption": (
            "within feature_space × MAD_decile stratum"
        ),
        "entropy": RNG_ENTROPY,
        "namespace": RNG_NAMESPACES[
            "primary_conditional_null"
        ],
        "generator": "PCG64DXSM",
        "seed_coordinates": (
            "namespace × replicate_id × stratum_id"
        ),
        "permutation_mapping": (
            "NumPy Generator.permutation"
        ),
        "coupling": (
            "complete three-program signed-weight vector "
            "permuted jointly within stratum"
        ),
        "draws_with_replacement": True,
        "h": MC_EXCEEDANCE_TARGET,
        "Bmax": FINAL_BMAX,
        "minimum_terminal_mc_p": (
            1.0
            / (
                FINAL_BMAX
                + 1.0
            )
        ),
        "batch_size": MC_BATCH_SIZE,
        "checkpoint_interval": (
            CHECKPOINT_INTERVAL
        ),
        "family_hash": (
            PRIMARY_FAMILY_HASH
        ),
        "observed_score_hash": (
            OBSERVED_PRIMARY_HASH
        ),
        "execution_spec_hash": (
            EXECUTION_SPEC_HASH
        ),
    },
    "multiplicity": {
        "primary": (
            "global Benjamini-Yekutieli"
        ),
        "q": PRIMARY_BY_Q,
        "family_size": PRIMARY_FAMILY_SIZE,
        "global_bh_role": (
            "nominal exploratory context only"
        ),
        "program_specific_adjustments_role": (
            "descriptive only"
        ),
    },
    "primary_status_counts": {
        str(key): int(value)
        for key, value in (
            primary_status_counts.items()
        )
    },
    "sensitivities": {
        "landmark_weighted_cosine": (
            "measured-only representation"
        ),
        "is_hiq_weighted_cosine": (
            "independent 5-cell-line / 4-lineage "
            "coverage requirement"
        ),
        "pooled_cell_line_weighted_cosine": (
            "composition sensitivity without equal-lineage weighting"
        ),
        "cmap_style_raw_wtcs": (
            "fixed 150 up / 150 down raw WTCS only"
        ),
        "competitive_matched_pseudoprogram": {
            "role": (
                "descriptive robustness analysis"
            ),
            "eligibility": (
                "PRIMARY_BY_SUPPORTED only"
            ),
            "draws": COMPETITIVE_DRAWS,
            "namespace": RNG_NAMESPACES[
                "competitive_null_sensitivity"
            ],
            "matching_plan_hash": (
                COMPETITIVE_PLAN_HASH
            ),
            "quantity": (
                "competitive_null_tail_fraction"
            ),
            "not_a_p_value": True,
        },
    },
    "diagnostics": {
        "provider_fields": list(
            PROVIDER_DIAGNOSTICS
        ),
        "hallmarks": list(
            FROZEN_HALLMARKS
        ),
        "hallmark_gene_set_metadata": (
            hallmark_diagnostic_metadata
            .to_dict(
                orient="records"
            )
        ),
        "role": (
            "contextual only; no exclusion or inferential gate"
        ),
    },
    "software": software_versions,
    "threadpool": threadpool_info(),
    "numerical_policy": {
        "source_dtype": "float32",
        "calculations": "float64",
        "rtol": NUMERICAL_RTOL,
        "atol": NUMERICAL_ATOL,
    },
    "stable_outputs": [
        "phase7.701.lineage_connectivity",
        "phase7.701.primary_perturbagen_connectivity",
        "phase7.701.sensitivity_connectivity",
        "phase7.701.perturbational_context_diagnostics",
        "phase7.701.analysis_metadata",
    ],
    "interpretation_limitations": [
        (
            "Conditional randomization support is conditional on "
            "the prospectively specified within-stratum "
            "exchangeability model."
        ),
        (
            "BY addresses multiplicity dependence conditional on "
            "valid marginal conditional-randomization p-values; "
            "it does not validate the scientific null."
        ),
        (
            "LINCS lineage balance reflects represented cell lines "
            "and does not establish pan-cancer population "
            "representativeness."
        ),
        (
            "Perturbational opposition is a computational association "
            "and does not establish therapeutic reversal, efficacy, "
            "causal mechanism, or clinical predictiveness."
        ),
        (
            "Sensitivity and contextual diagnostic results cannot "
            "rescue a primary result that fails the frozen global BY gate."
        ),
    ],
}



In [ ]:
# =============================================================================
# Persist Notebook 701 analysis metadata
# =============================================================================

with ANALYSIS_METADATA_PATH.open(
    "w",
    encoding="utf-8",
) as handle:
    json.dump(
        analysis_metadata,
        handle,
        indent=2,
        sort_keys=True,
    )
    handle.write("\n")

print("NOTEBOOK 701 STABLE OUTPUTS: PERSISTED")
print(
    "Sensitivity rows:",
    f"{len(sensitivity_connectivity):,}",
)
print(
    "Diagnostic rows:",
    f"{len(perturbational_context_diagnostics):,}",
)


In [ ]:
# =============================================================================
# Independently re-read and fingerprint stable Notebook 701 outputs
# =============================================================================

artifact_outputs = {
    "phase7.701.lineage_connectivity": {
        "path": LINEAGE_CONNECTIVITY_PATH,
        "format": "parquet",
        "artifact_role": "handoff",
    },
    "phase7.701.primary_perturbagen_connectivity": {
        "path": PRIMARY_CONNECTIVITY_PATH,
        "format": "csv",
        "artifact_role": "handoff",
    },
    "phase7.701.sensitivity_connectivity": {
        "path": SENSITIVITY_CONNECTIVITY_PATH,
        "format": "parquet",
        "artifact_role": "handoff",
    },
    "phase7.701.perturbational_context_diagnostics": {
        "path": CONTEXT_DIAGNOSTICS_PATH,
        "format": "parquet",
        "artifact_role": "handoff",
    },
    "phase7.701.analysis_metadata": {
        "path": ANALYSIS_METADATA_PATH,
        "format": "json",
        "artifact_role": "metadata",
    },
}

artifact_identity_rows = []

for artifact_id, spec in (
    artifact_outputs.items()
):
    artifact_path = spec[
        "path"
    ]

    if (
        not artifact_path.is_file()
        or artifact_path.stat().st_size
        == 0
    ):
        raise RuntimeError(
            f"Missing or empty stable output: {artifact_id}"
        )

    if spec["format"] == "parquet":
        persisted = pd.read_parquet(
            artifact_path
        )

        shape = [
            int(persisted.shape[0]),
            int(persisted.shape[1]),
        ]

    elif spec["format"] == "csv":
        persisted = pd.read_csv(
            artifact_path,
            low_memory=False,
        )

        shape = [
            int(persisted.shape[0]),
            int(persisted.shape[1]),
        ]

    elif spec["format"] == "json":
        with artifact_path.open(
            "r",
            encoding="utf-8",
        ) as handle:
            persisted = json.load(
                handle
            )

        if persisted != analysis_metadata:
            raise RuntimeError(
                "Persisted analysis metadata differs from in-memory metadata."
            )

        shape = None

    else:
        raise ValueError(
            f"Unknown output format: {spec['format']}"
        )

    artifact_identity_rows.append(
        {
            "artifact_id": artifact_id,
            "path": project_relative_path(
                artifact_path
            ),
            "artifact_role": spec[
                "artifact_role"
            ],
            "shape": shape,
            "size_bytes": int(
                artifact_path.stat().st_size
            ),
            "sha256": calculate_sha256(
                artifact_path
            ),
        }
    )

artifact_identities = pd.DataFrame(
    artifact_identity_rows
)

print(
    "NOTEBOOK 701 OUTPUT RE-READ: PASS"
)
display(artifact_identities)


In [ ]:
# =============================================================================
# Define Notebook 701 registry dependencies
# =============================================================================

NOTEBOOK_701_PATH = (
    "notebooks/phase7_perturbational_hypotheses/"
    "701_connectivity_analysis.ipynb"
)

artifact_inputs = {
    "phase7.701.lineage_connectivity": [
        {
            "type": "artifact",
            "artifact_id": (
                "phase7.700.program_query_definitions"
            ),
        },
        {
            "type": "artifact",
            "artifact_id": (
                "phase1.110.primary_signature_handoff"
            ),
        },
    ],
    "phase7.701.primary_perturbagen_connectivity": [
        {
            "type": "artifact",
            "artifact_id": (
                "phase7.701.lineage_connectivity"
            ),
        },
        {
            "type": "artifact",
            "artifact_id": (
                "phase7.700.null_stratification_manifest"
            ),
        },
        {
            "type": "artifact",
            "artifact_id": (
                "phase7.700.benchmark_report"
            ),
        },
        {
            "type": "artifact",
            "artifact_id": (
                "phase7.700.analysis_metadata"
            ),
        },
        {
            "type": "artifact",
            "artifact_id": (
                "phase1.110.primary_signature_handoff"
            ),
        },
    ],
    "phase7.701.sensitivity_connectivity": [
        {
            "type": "artifact",
            "artifact_id": (
                "phase7.701.primary_perturbagen_connectivity"
            ),
        },
        {
            "type": "artifact",
            "artifact_id": (
                "phase7.700.program_query_definitions"
            ),
        },
        {
            "type": "artifact",
            "artifact_id": (
                "phase7.700.null_stratification_manifest"
            ),
        },
        {
            "type": "artifact",
            "artifact_id": (
                "phase1.110.landmark_gene_handoff"
            ),
        },
        {
            "type": "artifact",
            "artifact_id": (
                "phase1.110.primary_signature_handoff"
            ),
        },
    ],
    "phase7.701.perturbational_context_diagnostics": [
        {
            "type": "artifact",
            "artifact_id": (
                "phase1.110.primary_signature_handoff"
            ),
        },
        {
            "type": "artifact",
            "artifact_id": (
                "phase7.700.program_query_definitions"
            ),
        },
        {
            "type": "raw_registry",
            "ref": (
                "/msigdb/files/"
                "h.all.v2026.1.Hs.symbols.gmt"
            ),
        },
    ],
    "phase7.701.analysis_metadata": [
        {
            "type": "artifact",
            "artifact_id": (
                "phase7.701.lineage_connectivity"
            ),
        },
        {
            "type": "artifact",
            "artifact_id": (
                "phase7.701.primary_perturbagen_connectivity"
            ),
        },
        {
            "type": "artifact",
            "artifact_id": (
                "phase7.701.sensitivity_connectivity"
            ),
        },
        {
            "type": "artifact",
            "artifact_id": (
                "phase7.701.perturbational_context_diagnostics"
            ),
        },
    ],
}



In [ ]:
# =============================================================================
# Register and validate frozen Notebook 701 artifacts
# =============================================================================

identity_index = (
    artifact_identities
    .set_index("artifact_id")
)

registry_entries = {}

for artifact_id in artifact_outputs:
    identity = identity_index.loc[
        artifact_id
    ]

    registry_entries[
        artifact_id
    ] = {
        "path": identity["path"],
        "phase": 7,
        "status": "frozen",
        "artifact_role": identity[
            "artifact_role"
        ],
        "producer": {
            "type": "notebook",
            "path": NOTEBOOK_701_PATH,
        },
        "shape": (
            identity["shape"]
            if isinstance(
                identity["shape"],
                list,
            )
            else None
        ),
        "size_bytes": int(
            identity["size_bytes"]
        ),
        "sha256": identity["sha256"],
        "inputs": artifact_inputs[
            artifact_id
        ],
    }

updated_registry = load_artifact_registry()

for artifact_id, entry in (
    registry_entries.items()
):
    existing = updated_registry[
        "artifacts"
    ].get(artifact_id)

    if (
        existing is not None
        and existing != entry
    ):
        raise RuntimeError(
            "A non-identical frozen Notebook 701 "
            f"registry entry already exists: {artifact_id}"
        )

    updated_registry[
        "artifacts"
    ][artifact_id] = entry

validate_artifact_registry(
    updated_registry
)

Paths.artifact_registry.write_text(
    json.dumps(
        updated_registry,
        indent=2,
        ensure_ascii=False,
    )
    + "\n",
    encoding="utf-8",
)

print(
    "NOTEBOOK 701 ARTIFACT REGISTRATION: PASS"
)
print(
    "Registered artifacts:",
    len(registry_entries),
)


In [ ]:
# =============================================================================
# Prepare final Notebook 701 closure QA
# =============================================================================

final_registry = load_artifact_registry()

registered_701 = {
    artifact_id: artifact
    for artifact_id, artifact in (
        final_registry[
            "artifacts"
        ].items()
    )
    if artifact_id.startswith(
        "phase7.701."
    )
}

allowed_primary_states = {
    "PRIMARY_BY_SUPPORTED",
    "INVERSE_NOT_BY_SUPPORTED",
    "NON_INVERSE",
    "NOT_EVALUABLE",
}

primary_p = (
    primary_perturbagen_connectivity[
        "p_for_multiplicity"
    ].to_numpy(dtype=np.float64)
)

final_primary_checkpoint = (
    read_primary_checkpoint(
        PRIMARY_CHECKPOINT_PATH,
        family_hash=PRIMARY_FAMILY_HASH,
        observed_hash=OBSERVED_PRIMARY_HASH,
    )
)



In [ ]:
# =============================================================================
# Evaluate final Notebook 701 closure conditions
# =============================================================================

qa_results = {
    "five_artifacts_registered": (
        set(registered_701)
        == set(artifact_outputs)
    ),
    "primary_family_23742": (
        len(
            primary_perturbagen_connectivity
        )
        == PRIMARY_FAMILY_SIZE
    ),
    "three_programs_complete": (
        primary_perturbagen_connectivity[
            "program_id"
        ]
        .value_counts()
        .reindex(
            PROGRAM_IDS
        )
        .eq(7914)
        .all()
    ),
    "primary_keys_unique": (
        not primary_perturbagen_connectivity
        .duplicated(
            [
                "program_id",
                "pert_id",
            ]
        )
        .any()
    ),
    "primary_states_valid": (
        set(
            primary_perturbagen_connectivity[
                "primary_support_status"
            ].unique()
        )
        <= allowed_primary_states
    ),
    "no_zero_primary_p": (
        np.all(
            primary_perturbagen_connectivity[
                "p_conditional"
            ].to_numpy(
                dtype=np.float64
            )
            > 0
        )
    ),
    "multiplicity_p_in_unit_interval": (
        np.all(
            (primary_p >= 0)
            & (primary_p <= 1)
        )
    ),
    "global_by_in_unit_interval": (
        primary_perturbagen_connectivity[
            "q_BY_global"
        ].between(
            0,
            1,
            inclusive="both",
        ).all()
    ),
    "global_bh_in_unit_interval": (
        primary_perturbagen_connectivity[
            "q_BH_global"
        ].between(
            0,
            1,
            inclusive="both",
        ).all()
    ),
    "noninverse_p_for_multiplicity_one": (
        primary_perturbagen_connectivity.loc[
            primary_perturbagen_connectivity[
                "directional_status"
            ].eq("NON_INVERSE"),
            "p_for_multiplicity",
        ]
        .eq(1.0)
        .all()
    ),
    "not_evaluable_p_for_multiplicity_one": (
        primary_perturbagen_connectivity.loc[
            primary_perturbagen_connectivity[
                "evaluability_status"
            ].eq("NOT_EVALUABLE"),
            "p_for_multiplicity",
        ]
        .eq(1.0)
        .all()
    ),
    "supported_rows_are_inverse": (
        primary_perturbagen_connectivity.loc[
            primary_perturbagen_connectivity[
                "primary_support_status"
            ].eq(
                "PRIMARY_BY_SUPPORTED"
            ),
            "directional_status",
        ]
        .eq("INVERSE")
        .all()
    ),
    "supported_rows_pass_global_by": (
        primary_perturbagen_connectivity.loc[
            primary_perturbagen_connectivity[
                "primary_support_status"
            ].eq(
                "PRIMARY_BY_SUPPORTED"
            ),
            "q_BY_global",
        ]
        .le(PRIMARY_BY_Q)
        .all()
    ),
    "lineage_scores_finite": (
        np.isfinite(
            lineage_connectivity[
                ["C_raw", "C_RL"]
            ].to_numpy(
                dtype=np.float64
            )
        ).all()
    ),
    "sensitivity_primary_rescue_absent": (
        "primary_support_status"
        not in sensitivity_connectivity.columns
    ),
    "diagnostics_have_no_inferential_p": (
        not any(
            column.startswith(
                "p_"
            )
            or column.startswith(
                "q_"
            )
            for column in (
                perturbational_context_diagnostics.columns
            )
        )
    ),
    "no_phase5_phase6_inputs": (
        not any(
            artifact_id.startswith(
                ("phase5.", "phase6.")
            )
            for artifact_id in (
                INPUT_ARTIFACT_IDS
            )
        )
    ),
    "final_bmax_exact": (
        FINAL_BMAX == 5058108
    ),
    "checkpoint_complete": (
        PRIMARY_CHECKPOINT_PATH.is_file()
        and not final_primary_checkpoint[
            "active_mask"
        ].any()
    ),
    "lineage_row_count": (
        len(lineage_connectivity)
        == 55827 * len(PROGRAM_IDS)
    ),
    "sensitivity_base_row_count": (
        len(sensitivity_connectivity)
        >= (
            4
            * PRIMARY_FAMILY_SIZE
        )
    ),
    "diagnostic_row_count": (
        len(
            perturbational_context_diagnostics
        )
        == (
            8
            * len(perturbagen_ids)
        )
    ),
}



In [ ]:
# =============================================================================
# Report final Notebook 701 closure status
# =============================================================================

for artifact_id, spec in (
    artifact_outputs.items()
):
    entry = registered_701[
        artifact_id
    ]

    qa_results[
        f"{artifact_id}:path"
    ] = (
        (
            Paths.root
            / entry["path"]
        ).resolve()
        == spec["path"].resolve()
    )

    qa_results[
        f"{artifact_id}:sha256"
    ] = (
        calculate_sha256(
            spec["path"]
        )
        == entry["sha256"]
    )

validate_artifact_registry(
    final_registry
)

qa_series = pd.Series(
    qa_results,
    name="PASS",
)

display(qa_series)

if not qa_series.all():
    failed = qa_series.loc[
        ~qa_series
    ].index.tolist()

    raise RuntimeError(
        "Notebook 701 closure QA failed: "
        + ", ".join(failed)
    )

print("NOTEBOOK 701 FINAL QA: PASS")
print(
    "Primary hypotheses:",
    f"{len(primary_perturbagen_connectivity):,}",
)
print(
    "PRIMARY_BY_SUPPORTED:",
    int(
        primary_perturbagen_connectivity[
            "primary_support_status"
        ].eq(
            "PRIMARY_BY_SUPPORTED"
        ).sum()
    ),
)
print(
    "Stable outputs registered:",
    "5/5",
)


## Notebook completion

Notebook 701 is complete only when the final QA cell passes.

Primary inference remains defined at the perturbagen level under the frozen conditional assignment-specific randomization model and global BY family. Sensitivities and perturbational-context diagnostics remain secondary/contextual and cannot rescue primary inference.

Any inverse perturbational association identified here is a computational perturbational hypothesis. It is not evidence of therapeutic reversal, treatment efficacy, causal mechanism, a validated target, or a clinical predictor.
